# DSCxxxx — lagi cape lomba asli

**Anggota (akun Kaggle):** athillazaidan, nicholaswisee, riantamaputra

_Nomor peserta DSCxxxx masih placeholder; nama file wajib `ParticipantNumber_TeamName_Notebook`._

# Credit risk: merged team pipeline

DSC LOGIKA UI 2026. Equal-weight rank blend of the three team pipelines, all validated on one fold scheme: **R2** (teman goal054: Kumo V5 ctx50k e8 + offset re_* + spline 25%, model seeds 2026 and 2027 averaged (R2)), **D** (DSC0152 v66 ext0.5|tm30|lgb_lean, 5-fold seed 42, inner early stop + refit), **K64** (Kumo strict (utama): cap300@28k + main_small, 2 seeds, pools fit on outer-train). Organizer data only; public pretrained Kumo-Tabular v1.0.1 weights.

## 1. Validation evidence

StratifiedKFold 5, shuffle, seed 42 (`eda/valid2.outer`); every family is out-of-fold on these folds; feature references, early stopping and corrections are fit inside the outer-training partition. Blend weights were locked in protocol JSON files before the outer scores were read. Pooled OOF KS (70,000 rows, rank per outer cohort) **0.52886**, AUC 0.84861, fold KS 0.5292, 0.5362, 0.5268, 0.5153, 0.5520. These are validation estimates, not leaderboard claims.

## 2. Settings

Run from the repository (or set LOGIKA_REPOSITORY). Default `replay` checks the embedded sources against the repository, re-blends the cached family predictions and checks the CSV hash against the reference manifest. `LOGIKA_PIPELINE_MODE=fresh` first runs every missing family build command of the recipe (hours of GPU/CPU, one process at a time).

In [ ]:
import os,sys,json,hashlib,subprocess
from pathlib import Path
import numpy as np
import polars as pl
MODE=os.environ.get('LOGIKA_PIPELINE_MODE','replay')
assert MODE in ('replay','fresh')
REPOSITORY=Path(os.environ.get('LOGIKA_REPOSITORY',str(Path.cwd()))).resolve()
if not (REPOSITORY/'data').exists() and (REPOSITORY.parent/'data').exists(): REPOSITORY=REPOSITORY.parent
OUTPUT_DIR=Path(os.environ.get('LOGIKA_OUTPUT_DIR',str(REPOSITORY/'results/v71/merge_pipeline/notebook_replay'))).resolve()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','POLARS_MAX_THREADS'): os.environ[name]='4'
print('mode',MODE,'output',OUTPUT_DIR)

## 3. Source

The blend, the recipe and the family runners are embedded below with SHA256 and must equal the repository copies that are executed.

In [ ]:
FILES={'eda/260_merge_pipeline.py': '"""Single entry point of the merged team pipeline (utama + teman goal054 + DSC0152) on StratifiedKFold 5 seed 42.\n\nRecipe: artifacts/optimization/continuation_25okt/recipe_merge.json (families, their build commands/outputs, blend).\n  --stage plan            list every build command and whether its outputs are cached\n  --stage build [--family X]  run missing commands in order, one process at a time (start eda/thermal_guard.py first)\n  --stage blend           locked equal-weight per-cohort rank blend -> OOF report, submission.csv, manifest (SHA256)\n  --stage verify          re-run blend in memory and assert submission/OOF bytes match the stored manifest\nUsage: cd eda && ../.venv/bin/python 260_merge_pipeline.py --stage plan\n"""\nimport argparse\nimport hashlib\nimport importlib\nimport io\nimport json\nimport os\nimport subprocess\n\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom common import ROOT\n\nRECIPE = ROOT / "artifacts/optimization/continuation_25okt/recipe_merge.json"\ne250 = importlib.import_module("250_merge_blend_existing")\n\n\ndef sha(path):\n    with open(path, "rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef write_atomic(path, data):\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with open(tmp, "wb") as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef plan(recipe, family=None):\n    for name, fam in recipe["families"].items():\n        if family and name != family:\n            continue\n        for step in fam["steps"]:\n            done = all((ROOT / p).exists() for p in step["outputs"])\n            yield name, step, done\n\n\ndef blend(recipe):\n    ids, y, tids, folds, fam_oof, fam_test, _ = e250.families(str(ROOT / recipe["dsc_npz"]))\n    members = recipe["blend"]["members"]\n    oof = np.zeros(len(y))\n    for _, va in folds:  # per-cohort rank inside each outer fold, equal weights (locked)\n        oof[va] = np.mean([e250.rank(fam_oof[m][va]) for m in members], axis=0)\n    test = np.mean([e250.rank(fam_test[m]) for m in members], axis=0)\n    sub = pl.DataFrame({"application_id": tids, "default_probability": test})\n    assert sub.height == 30000 and sub["application_id"].n_unique() == 30000\n    assert np.isfinite(test).all() and test.min() >= 0 and test.max() <= 1\n    report = {"members": members, "pooled_ks": e250.ks(y, oof), "auc": float(roc_auc_score(y, oof)),\n              "fold_ks": [e250.ks(y[va], oof[va]) for _, va in folds]}\n    return sub, pl.DataFrame({"application_id": ids, "target": y, "oof": oof}), report\n\n\ndef to_bytes(frame, kind):\n    buf = io.BytesIO()\n    frame.write_csv(buf) if kind == "csv" else frame.write_parquet(buf)\n    return buf.getvalue()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--stage", required=True, choices=["plan", "build", "blend", "verify"])\n    ap.add_argument("--family")\n    ap.add_argument("--recipe", default=str(RECIPE))\n    ap.add_argument("--out", help="output folder (default: recipe output_dir)")\n    a = ap.parse_args()\n    recipe = json.loads(open(a.recipe).read())\n    out = ROOT / (a.out or recipe["output_dir"])\n    if a.stage == "plan":\n        for name, step, done in plan(recipe, a.family):\n            print(f"[{\'cached\' if done else \'MISSING\'}] {name}: {step[\'cmd\']}")\n    elif a.stage == "build":\n        for name, step, done in plan(recipe, a.family):\n            if not done:\n                print(f"=== {name}: {step[\'cmd\']}", flush=True)\n                subprocess.run(step["cmd"], shell=True, check=True, cwd=ROOT / step.get("cwd", "eda"))\n    else:\n        missing = [n for n, s, d in plan(recipe) if not d and n in recipe["blend"]["members"]]\n        assert not missing, f"families not built: {missing}"\n        sub, oof, report = blend(recipe)\n        files = {"submission.csv": to_bytes(sub, "csv"), "oof.parquet": to_bytes(oof, "parquet")}\n        if a.stage == "verify":\n            manifest = json.loads((out / "manifest.json").read_text())\n            for fname, data in files.items():\n                assert hashlib.sha256(data).hexdigest() == manifest["outputs"][fname], fname\n            print("verify OK", manifest["outputs"]["submission.csv"])\n            return\n        out.mkdir(parents=True, exist_ok=True)\n        for fname, data in files.items():\n            write_atomic(out / fname, data)\n        inputs = sorted({p for _, s, _ in plan(recipe) for p in s["outputs"]\n                         if any(p.startswith(x) for x in recipe["blend"]["input_prefixes"])})\n        manifest = {"recipe": recipe, "report": report, "outputs": {f: sha(out / f) for f in files},\n                    "inputs": {p: sha(ROOT / p) for p in inputs if (ROOT / p).is_file()}}\n        write_atomic(out / "manifest.json", json.dumps(manifest, indent=2).encode())\n        print(json.dumps(report), "\\nsubmission", manifest["outputs"]["submission.csv"])\n\n\nif __name__ == "__main__":\n    main()\n', 'eda/250_merge_blend_existing.py': '"""Merge step (i): equal-weight rank blends of existing OOF families across the three team pipelines.\n\nFamilies (protocol artifacts/optimization/continuation_25okt/protocol_250_blend_existing.json, locked before scores):\nT trees6 v22, C Causilo, KL Kumo legacy v28m, K64 Kumo strict 2-seed, D = DSC0152 v66 ext0.5|tm30|lgb_lean.\nD is OOF averaged over 10-fold x 6 fold seeds -> exploratory until reproduced on seed-42 5-fold.\nPrints pooled/fold KS+AUC, bootstrap deltas vs REF and vs best component, test Spearman vs scored submissions.\nUsage: cd eda && ../.venv/bin/python 250_merge_blend_existing.py [--d-npz path] [--tag name]\n"""\nimport importlib\nimport json\nimport sys\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport polars as pl\nfrom scipy.stats import rankdata, spearmanr\nfrom sklearn.metrics import roc_auc_score\n\nimport valid2 as v\nfrom common import ROOT\n\ne221 = importlib.import_module("221_evaluate_recipe_candidates")\nrank, load, fast_ks, ks, R = e221.rank, e221.load, e221.fast_ks, e221.ks, e221.R\nPROTO = ROOT / "artifacts/optimization/continuation_25okt/protocol_250_blend_existing.json"\nSCORED = {"goal054_0.56470": R / "v70/goal054_kumo_offset/submission.csv", "v31_0.56405": R / "v31/submission.csv",\n          "K64_0.56168": R / "v63/v70b_k64_2seed/submission.csv"}\n\n\ndef pd_read(path):\n    return pl.read_parquet(path)\n\n\ndef dsc_recipe(z, prefix):\n    """v66 make_submission formula: ext0.5 of tm30 away from lgb_lean (ranks over all rows of the array)."""\n    r = {k[len(prefix):]: rank(z[k]) for k in z.files if k.startswith(prefix)}\n    trees = (r["lgb_full"] + r["lgb_lean"] + r["cat_full"] + r["xgb_full"]) / 4\n    tm30 = 0.7 * (trees * 2 + r["lgbx_full"]) / 3 + 0.3 * r["linear"]\n    return rank(tm30 + 0.5 * (tm30 - r["lgb_lean"]))\n\n\ndef by_id(values, src_ids, dst_ids):\n    pos = {t: i for i, t in enumerate(src_ids)}\n    return values[np.array([pos[t] for t in dst_ids])]\n\n\ndef families(d_npz):\n    """All seed-42 family OOF (by sorted id) and test vectors (sample order) + the DSC npz."""\n    train = pl.read_csv(ROOT / "data/train_applications.csv").sort("application_id")\n    ids, y = train["application_id"].to_numpy(), train["target"].to_numpy().astype(int)\n    tids = pl.read_csv(ROOT / "data/sample_submission.csv")["application_id"].to_numpy()  # sample order\n    stids = np.sort(tids)\n    folds = v.outer(y, 42)\n    fam_oof, fam_test = {}, {}\n    fam_oof["T"] = np.mean([rank(load(str(p).format("oof"), ids)) for p in e221.TREES], axis=0)\n    fam_test["T"] = by_id(np.mean([rank(load(str(p).format("test"), stids)) for p in e221.TREES], axis=0), stids, tids)\n    fam_oof["C"] = sum(w * load(str(p).format("oof"), ids) for w, p in e221.CAUSILO) / 5\n    fam_test["C"] = by_id(sum(w * load(str(p).format("test"), stids) for w, p in e221.CAUSILO) / 5, stids, tids)\n    fam_oof["KL"] = load(R / "v28m/oof_kumo_large_c56000.parquet", ids)\n    k64 = pl.read_parquet(R / "v60/candidates/K__K64_2seed/oof.parquet").sort("application_id")\n    assert k64["application_id"].to_list() == ids.tolist()\n    fam_oof["K64"] = k64["K"].to_numpy()\n    k64t = pl.read_csv(R / "v60/candidates/K__K64_2seed/submission.csv")\n    fam_test["K64"] = by_id(k64t["default_probability"].to_numpy(), k64t["application_id"].to_numpy(), tids)\n    z = np.load(d_npz, allow_pickle=True)\n    fam_oof["D"] = by_id(dsc_recipe(z, "oof_"), z["train_ids"], ids)\n    fam_test["D"] = by_id(dsc_recipe(z, "test_"), z["test_ids"], tids)\n    r_dir = R / "v71/teammate_s42"\n    for fam, stem in [("R", ""), ("KF", "_kf"), ("KD", "_kd"), ("T5", "_t5")]:  # teammate recipe on seed 42 (eda/251), Kumo V5+factor (eda/253)\n        if (r_dir / f"oof{stem}.parquet").exists():\n            ro, rt = pd_read(r_dir / f"oof{stem}.parquet"), pd_read(r_dir / f"test{stem}.parquet")\n            fam_oof[fam] = by_id(ro[fam].to_numpy(), ro["application_id"].to_numpy(), ids)\n            fam_test[fam] = by_id(rt[fam].to_numpy(), rt["application_id"].to_numpy(), tids)\n    if "R" in fam_oof and (r_dir / "oof_s2027.parquet").exists():  # R2 = mean of two model seeds (protocol 256)\n        ro, rt = pd_read(r_dir / "oof_s2027.parquet"), pd_read(r_dir / "test_s2027.parquet")\n        fam_oof["R2"] = (fam_oof["R"] + by_id(ro["R"].to_numpy(), ro["application_id"].to_numpy(), ids)) / 2\n        fam_test["R2"] = (fam_test["R"] + by_id(rt["R"].to_numpy(), rt["application_id"].to_numpy(), tids)) / 2\n    if "KD" in fam_oof and (r_dir / "oof_kd_s2027.parquet").exists():  # KD2 = mean of two model seeds (protocol 257)\n        ro, rt = pd_read(r_dir / "oof_kd_s2027.parquet"), pd_read(r_dir / "test_kd_s2027.parquet")\n        fam_oof["KD2"] = (fam_oof["KD"] + by_id(ro["KD"].to_numpy(), ro["application_id"].to_numpy(), ids)) / 2\n        fam_test["KD2"] = (fam_test["KD"] + by_id(rt["KD"].to_numpy(), rt["application_id"].to_numpy(), tids)) / 2\n    for k in ["lgb_lean", "xgb_full", "fac_full"]:\n        fam_oof[f"D_{k}"] = by_id(z[f"oof_{k}"], z["train_ids"], ids)\n    return ids, y, tids, folds, fam_oof, fam_test, z\n\n\ndef main():\n    args = sys.argv[1:]\n    d_npz = args[args.index("--d-npz") + 1] if "--d-npz" in args else str(R / "dsc0152/agas/v66/output/preds_v66.npz")\n    tag = args[args.index("--tag") + 1] if "--tag" in args else "v66"\n    proto_path = ROOT / args[args.index("--protocol") + 1] if "--protocol" in args else PROTO\n    proto = json.loads(proto_path.read_text())\n    ids, y, tids, folds, fam_oof, fam_test, z = families(d_npz)\n    cands = {"REF": ["T", "C", "KL"], **proto.get("candidates_equal_weight_rank", proto.get("candidates", {}))}\n    cands = {n: ms for n, ms in cands.items() if all(m in fam_oof for m in ms)}  # skip families not built yet\n    oof = {}\n    for n, members in cands.items():\n        oof[n] = np.full(len(y), np.nan)\n        for tr, va in folds:  # per-cohort rank inside each outer fold\n            oof[n][va] = np.mean([rank(fam_oof[m][va]) for m in members], axis=0)\n    test = {n: np.mean([rank(fam_test[m]) for m in ms], axis=0) for n, ms in cands.items() if "KL" not in ms}\n    rng = np.random.default_rng(250)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    boots = [np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(1000)]\n    comp = {}  # components ranked per cohort, like the blends, so deltas compare like with like\n    for m in [m for m in ["T", "C", "KL", "K64", "D", "R", "R2", "KF", "KD", "KD2", "T5"] if m in fam_oof]:\n        comp[m] = np.full(len(y), np.nan)\n        for tr, va in folds:\n            comp[m][va] = rank(fam_oof[m][va])\n    print("components pooled KS / AUC (raw, global):")\n    for m, p in fam_oof.items():\n        print(f"  {m:12s} {ks(y, p):.5f} {roc_auc_score(y, p):.5f} folds",\n              [round(ks(y[va], p[va]), 4) for _, va in folds])\n    bks = {n: fast_ks(y, p, boots) for n, p in {**oof, **comp}.items()}\n    report = {"protocol": proto["id"], "d_npz": d_npz, "candidates": {}}\n    for n, members in cands.items():\n        r = {"members": members, "pooled_ks": ks(y, oof[n]), "auc": float(roc_auc_score(y, oof[n])),\n             "fold_ks": [ks(y[va], oof[n][va]) for _, va in folds]}\n        best = max(members, key=lambda m: ks(y, comp[m]))\n        for base_name, base in [("REF", oof["REF"]), (f"best_{best}", comp[best])]:\n            if n == "REF":\n                continue\n            key = "REF" if base_name == "REF" else best\n            d = bks[n] - bks[key]\n            r[f"delta_vs_{base_name}"] = {"observed": r["pooled_ks"] - ks(y, base),\n                                          "ci95": np.quantile(d, [.025, .975]).tolist(), "p_gt0": float((d > 0).mean()),\n                                          "folds_won": int(sum(ks(y[va], oof[n][va]) > ks(y[va], base[va]) for _, va in folds))}\n        if n in test:\n            r["test_spearman"] = {}\n            for s, path in SCORED.items():\n                if path.exists():\n                    sub = pl.read_csv(path)\n                    r["test_spearman"][s] = float(spearmanr(test[n], by_id(sub["default_probability"].to_numpy(),\n                                                                         sub["application_id"].to_numpy(), tids))[0])\n        report["candidates"][n] = r\n        print(n, json.dumps({k: (round(x, 5) if isinstance(x, float) else x) for k, x in r.items()\n                             if k not in ("fold_ks", "members")}, default=str),\n              "folds", [round(x, 4) for x in r["fold_ks"]])\n    out = ROOT / f"eda/out/250_blend_existing_{tag}.json"\n    out.write_text(json.dumps(report, indent=2))\n    folder = R / f"v71/merge_blend/{tag}"\n    folder.mkdir(parents=True, exist_ok=True)\n    for n, t in test.items():\n        sub = pl.DataFrame({"application_id": tids, "default_probability": t})\n        assert sub.height == 30000 and sub["application_id"].n_unique() == 30000 and t.min() >= 0 and t.max() <= 1\n        sub.write_csv(folder / f"{n}.csv")\n    pl.DataFrame({"application_id": ids, "target": y, **oof}).write_parquet(folder / "oof.parquet")\n    fig, ax = plt.subplots(figsize=(8, 4))\n    for n, r in report["candidates"].items():\n        ax.plot(range(5), r["fold_ks"], marker="o", label=f"{n} {r[\'pooled_ks\']:.4f}")\n    ax.set_xlabel("outer fold (seed 42)"); ax.set_ylabel("KS"); ax.legend(fontsize=7); ax.set_title(f"250 merge blends ({tag})")\n    fig.tight_layout(); fig.savefig(ROOT / f"eda/out/250_blend_existing_{tag}.png", dpi=120)\n\n\nif __name__ == "__main__":\n    main()\n', 'eda/221_evaluate_recipe_candidates.py': '"""Evaluate locked candidates of amendment_01 from cached strict members (complete 5-fold members only).\n\nK   = equal rank mean of strict Kumo members (per outer cohort, test per fold then mean over folds)\nKT  = 0.5 rank(K) + 0.5 rank(trees6)            (v22 GBDT family)\nKTC = thirds rank(trees6), rank(Causilo), rank(K)  (v59 structure, legacy Kumo replaced)\nREF = thirds trees6 / Causilo / legacy Kumo v28m (OOF analogue of v59; public 0.56296)\nWeights are fixed by amendment_01; nothing is tuned on OOF. Prints pooled/fold KS+AUC, paired\nstratified bootstrap deltas, writes oof/test/submission per candidate and one figure.\nUsage: cd eda && ../.venv/bin/python 221_evaluate_recipe_candidates.py [member ...]\n"""\nimport hashlib\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport polars as pl\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\n\nimport valid2 as v\nfrom common import ROOT\n\nSTRICT = ROOT / "results/v60/strict_tfm"\nOUT = ROOT / "results/v60/candidates"\nR = ROOT / "results"\nTREES = [R / f"v22/v22/gbdt/{{}}_{m}.parquet" for m in ["lgb", "lgb_s", "xgb", "xgb_s", "cat", "cat_s"]]\nCAUSILO = [(1, R / "v22/v22/fm/{}_causilo.parquet"), (2, R / "v37/{}_causilo.parquet"), (2, R / "v41/causilo_s12/{}_causilo.parquet")]\nDEFAULT_MEMBERS = [f"{view}_s{seed}" for view in ("pool_cap500", "nested_small", "pool_uncapped") for seed in (2026, 2027)]\n\n\ndef sha(path):\n    with open(path, "rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef rank(x):\n    return rankdata(x, method="average") / len(x)\n\n\ndef ks(y, p):\n    return v.ks(y, p)\n\n\ndef fast_ks(y, p, rows_list):\n    """KS on bootstrap index sets via sorted order (ties broken consistently by stable sort)."""\n    out = []\n    for rows in rows_list:\n        yy, pp = y[rows], p[rows]\n        order = np.argsort(-pp, kind="stable")\n        ys, ps = yy[order], pp[order]\n        tp, fp = np.cumsum(ys), np.cumsum(1 - ys)\n        last = np.r_[ps[1:] != ps[:-1], True]  # evaluate only at distinct-score boundaries\n        out.append(np.max(np.abs(tp[last] / tp[-1] - fp[last] / fp[-1])))\n    return np.array(out)\n\n\ndef load(path, ids):\n    d = pl.read_parquet(path).sort("application_id")\n    assert d["application_id"].to_list() == sorted(ids), path\n    x = d[[c for c in d.columns if c not in ("application_id", "target")][0]].to_numpy()\n    assert np.isfinite(x).all(), path\n    return x\n\n\ndef main():\n    args = sys.argv[1:]\n    tag = args.pop(args.index("--tag") + 1) if "--tag" in args else None\n    if tag:\n        args.remove("--tag")\n    members = args or DEFAULT_MEMBERS\n    train = pl.read_csv(ROOT / "data/train_applications.csv").sort("application_id")\n    ids, y = train["application_id"].to_numpy(), train["target"].to_numpy().astype(int)\n    tids = pl.read_csv(ROOT / "data/sample_submission.csv")["application_id"].to_numpy()\n    stids = np.sort(tids)\n    folds = v.outer(y, 42)\n    available = []\n    roots = {}\n    for i, m in enumerate(members):  # "results/v62/strict_variant::member" reads another root\n        root, m = (ROOT / m.split("::")[0], m.split("::")[1]) if "::" in m else (STRICT, m)\n        members[i] = m\n        roots[m] = root\n        ok = all((root / f"outer_f{f}/{m}.json").exists() for f in range(5))\n        print(f"member {m}: {\'complete\' if ok else \'INCOMPLETE -> excluded\'}")\n        if ok:\n            available.append(m)\n    assert available, "no complete member"\n    # legacy/historical families (OOF averaged over outer seeds, already OOF per row)\n    trees_oof = np.mean([rank(load(str(p).format("oof"), ids)) for p in TREES], axis=0)\n    trees_test = np.mean([rank(load(str(p).format("test"), tids)) for p in TREES], axis=0)\n    cau_oof = sum(w * load(str(p).format("oof"), ids) for w, p in CAUSILO) / 5\n    cau_test = sum(w * load(str(p).format("test"), tids) for w, p in CAUSILO) / 5\n    kumo_legacy_oof = load(R / "v28m/oof_kumo_large_c56000.parquet", ids)\n    member_oof = {m: np.full(len(y), np.nan) for m in available}\n    names = ["K", "KT", "KTC", "REF"]\n    oof = {n: np.full(len(y), np.nan) for n in names}\n    test_parts = {n: [] for n in ["K", "KT", "KTC"]}\n    test_pos = {t: i for i, t in enumerate(stids)}\n    evidence = []\n    for f, (tr, va) in enumerate(folds):\n        val_ranks, test_ranks = [], []\n        for m in available:\n            folder = roots[m] / f"outer_f{f}"\n            manifest = json.loads((folder / "manifest.json").read_text())\n            assert manifest["fit_ids"] == ids[tr].tolist() and manifest["query_ids"] == ids[va].tolist()\n            assert not set(manifest["fit_ids"]) & set(manifest["query_ids"])\n            meta = json.loads((folder / f"{m}.json").read_text())\n            pp = folder / f"{m}.parquet"\n            assert meta["manifest_sha256"] == sha(folder / "manifest.json") and meta["prediction_sha256"] == sha(pp)\n            assert meta["metadata"]["query_labels_accessed"] is False\n            d = pl.read_parquet(pp)\n            assert d["application_id"].to_list() == ids[va].tolist() + tids.tolist()\n            p = d["prediction"].to_numpy()\n            member_oof[m][va] = p[:len(va)]\n            val_ranks.append(rank(p[:len(va)]))\n            test_ranks.append(rank(p[len(va):]))\n            evidence.append({"fold": f, "member": m, "prediction_sha256": meta["prediction_sha256"],\n                             "context": meta["metadata"]["context_count"],\n                             "processed_columns": [c["columns"] for c in meta["metadata"]["processed_contexts"]][:1],\n                             "memory_fraction": meta["metadata"]["memory_fraction"]})\n        k_val, k_test = np.mean(val_ranks, axis=0), np.mean(test_ranks, axis=0)\n        t_val, c_val, l_val = rank(trees_oof[va]), rank(cau_oof[va]), rank(kumo_legacy_oof[va])\n        oof["K"][va] = k_val\n        oof["KT"][va] = (rank(k_val) + t_val) / 2\n        oof["KTC"][va] = (rank(k_val) + t_val + c_val) / 3\n        oof["REF"][va] = (l_val + t_val + c_val) / 3\n        # test ids in sample order -> map trees/causilo (sorted order) into sample order\n        order = np.array([test_pos[t] for t in tids])\n        kt = rank(k_test)\n        test_parts["K"].append(k_test)\n        test_parts["KT"].append((kt + rank(trees_test[order])) / 2)\n        test_parts["KTC"].append((kt + rank(trees_test[order]) + rank(cau_test[order])) / 3)\n    assert all(np.isfinite(o).all() for o in oof.values())\n    rng = np.random.default_rng(221)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    boots = [np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(1000)]\n    bks = {n: fast_ks(y, oof[n], boots) for n in names}\n    report = {"members_used": available, "members_requested": members, "evidence": evidence, "candidates": {}}\n    for m in available:\n        report.setdefault("member_pooled_ks", {})[m] = ks(y, member_oof[m])\n    for n in names:\n        r = {"pooled_ks": ks(y, oof[n]), "auc": float(roc_auc_score(y, oof[n])),\n             "fold_ks": [ks(y[va], oof[n][va]) for _, va in folds]}\n        r["mean_fold_ks"] = float(np.mean(r["fold_ks"]))\n        for base in ["REF", "K"]:\n            if base != n:\n                d = bks[n] - bks[base]\n                r[f"delta_vs_{base}"] = {"observed": r["pooled_ks"] - ks(y, oof[base]),\n                                        "ci95": np.quantile(d, [.025, .975]).tolist(), "sd": float(d.std(ddof=1)),\n                                        "p_gt0": float((d > 0).mean()),\n                                        "folds_won": int(sum(ks(y[va], oof[n][va]) > ks(y[va], oof[base][va]) for _, va in folds))}\n        report["candidates"][n] = r\n        print(n, json.dumps({k: (round(x, 5) if isinstance(x, float) else x) for k, x in r.items() if k != "fold_ks"}),\n              "folds", [round(x, 4) for x in r["fold_ks"]])\n    for m, s in report.get("member_pooled_ks", {}).items():\n        print(f"  member {m}: pooled KS {s:.5f}")\n    tag = tag or f"{len(available)}m"\n    names_used = "+".join(available)\n    for n in ["K", "KT", "KTC"]:\n        folder = OUT / f"{n}__{tag}"\n        folder.mkdir(parents=True, exist_ok=True)\n        pred = np.mean(test_parts[n], axis=0)\n        sub = pl.DataFrame({"application_id": tids, "default_probability": pred})\n        assert sub.height == 30000 and sub["application_id"].n_unique() == 30000 and set(sub["application_id"]) == set(tids)\n        assert np.isfinite(pred).all() and pred.min() >= 0 and pred.max() <= 1\n        tmp = folder / "submission.csv.tmp"\n        sub.write_csv(tmp)\n        with open(tmp, "rb") as s:\n            os.fsync(s.fileno())\n        tmp.replace(folder / "submission.csv")\n        pl.DataFrame({"application_id": ids, "target": y, n: oof[n]}).write_parquet(folder / "oof.parquet")\n        (folder / "report.json").write_text(json.dumps({"members": names_used, "candidate": n, **report["candidates"][n],\n                                                        "submission_sha256": sha(folder / "submission.csv")}, indent=2))\n    (OUT / f"report__{tag}.json").write_text(json.dumps(report, indent=2))\n    fig, ax = plt.subplots(figsize=(7, 4))\n    for n in names:\n        ax.plot(range(5), report["candidates"][n]["fold_ks"], marker="o", label=f"{n} {report[\'candidates\'][n][\'pooled_ks\']:.4f}")\n    ax.set_xlabel("outer fold (seed 42)"); ax.set_ylabel("KS"); ax.legend(); ax.set_title(f"221 candidates, members: {len(available)}")\n    fig.tight_layout(); fig.savefig(ROOT / f"eda/out/221_candidates_{tag}.png", dpi=120)\n\n\nif __name__ == "__main__":\n    main()\n', 'eda/251_teammate_recipe_s42.py': '"""Reproduce teammate goal054 (Kumo V5 + latent offset + spline) on StratifiedKFold 5 seed 42.\n\nProtocol: artifacts/optimization/continuation_25okt/protocol_251_teammate_s42.json (locked before scores).\nStages per fold (each cached atomically, one heavy unit per process):\n  reverse  re_* latent features from outer-train customers only (label-free)       [CPU]\n  kumo     Kumo large, 90 V5 cols, context 50k (teammate sampler), e8, seed 2026+fold [GPU]\n  post     offset correction on the 6k holdout + spline on 56k outer-train          [CPU]\n  eval     pooled OOF of K5, K5o, S, R0, R over the 5 folds\nUsage: cd eda && ../.venv/bin/python 251_teammate_recipe_s42.py --stage {reverse,kumo,post} --fold k | --stage eval\n"""\nimport argparse\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nimport valid2 as v\nfrom common import ROOT\n\nsys.path.insert(0, str(ROOT / "scripts"))\nTEAM = ROOT / "results/v56/teammate_features"\nOUT = Path(os.environ.get("LOGIKA_MERGE_OUT", ROOT / "results/v71/teammate_s42"))  # override = fresh reproduction check\nDEF = ROOT / "artifacts/optimization/goal054/cv5_seed31027/kumo_offset_l20p003_unweighted_d0p5/definition.json"\nSPEC = dict(penalty=0.003, balance="unweighted", correction_strength=0.5)\nSEED = 2026  # base of the context/model seed (seed + fold); --seed 2027 = second seed (protocol 256)\n\n\ndef sfx():\n    return "" if SEED == 2026 else f"_s{SEED}"\n\n\ndef atomic_parquet(frame, path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    buf = io.BytesIO()\n    frame.to_parquet(buf, index=False)\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "wb") as f:\n        f.write(buf.getvalue())\n        f.flush()\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef atomic_json(value, path):\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "w") as f:\n        f.write(json.dumps(value, indent=2, default=str))\n        f.flush()\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef labels():\n    train = pd.read_csv(ROOT / "data/train_applications.csv", usecols=["application_id", "target"])\n    return train.sort_values("application_id").reset_index(drop=True)\n\n\ndef fold_frame(fold):\n    """V5 (+ re_* when built) for the fold; val/test rows carry no label. Returns frame, fit/val/test ids."""\n    v5 = pd.read_parquet(TEAM / f"outer_s42_f{fold}/data/processed/application_features_v5.parquet")\n    lab = labels()\n    folds = v.outer(lab.target.to_numpy(), 42)\n    tr, va = folds[fold]\n    fit_ids, val_ids = set(lab.application_id.iloc[tr]), set(lab.application_id.iloc[va])\n    assert set(v5.application_id[v5.split.eq("train")]) == fit_ids, "fold dir fitting ids != valid2.outer seed 42"\n    assert v5.target[~v5.split.eq("train")].isna().all()\n    rev = OUT / f"f{fold}/reverse.parquet"\n    if rev.exists():\n        v5 = v5.merge(pd.read_parquet(rev), on="application_id", how="left", validate="one_to_one")\n    return v5, fit_ids, val_ids\n\n\ndef stage_reverse(fold):\n    import reverse_synthetic_patterns as rsp\n\n    v5, _, _ = fold_frame(fold)\n    rsp.ROOT = TEAM / f"outer_s42_f{fold}"  # raw csv symlinks there; split==\'train\' = outer-train customers\n    rsp.distribution_summary = lambda values: {}  # audit-only summary; its scipy kstest call breaks on scipy here\n    base = v5.set_index("customer_id")\n    parts = [rsp.fit_price_latent(base)[0], rsp.fit_bureau_latent(base)[0], rsp.fit_account_latent(base)[0],\n             rsp.fit_repayment_latent(base)[0]]\n    new = pd.concat(parts, axis=1)\n    assert new.index.equals(base.index) and new.columns.is_unique and not any("target" in c for c in new.columns)\n    new = new.replace([np.inf, -np.inf], np.nan)\n    new.insert(0, "application_id", base.application_id.to_numpy())\n    atomic_parquet(new.reset_index(drop=True), OUT / f"f{fold}/reverse.parquet")\n    print("reverse", fold, new.shape, flush=True)\n\n\ndef split_rows(frame, fit_ids, val_ids, fold):\n    from kumo_tabular import context_indices\n\n    lab = labels().set_index("application_id").target\n    is_fit = frame.application_id.isin(fit_ids).to_numpy()\n    ti = np.flatnonzero(is_fit)\n    y = np.zeros(len(frame), dtype=int)\n    y[ti] = lab.loc[frame.application_id.iloc[ti]].to_numpy()\n    ci = np.sort(context_indices(ti, y, 50000, SEED + fold))\n    hi = np.setdiff1d(ti, ci)\n    vi = np.flatnonzero(frame.application_id.isin(val_ids).to_numpy())\n    te = np.flatnonzero(~is_fit & ~frame.application_id.isin(val_ids).to_numpy())\n    assert len(ci) == 50000 and len(hi) == 6000 and len(vi) == 14000 and len(te) == 30000\n    return y, ti, ci, hi, vi, te\n\n\ndef stage_kumo(fold):\n    import strict_kumo_engine as eng\n\n    frame, fit_ids, val_ids = fold_frame(fold)\n    y, ti, ci, hi, vi, te = split_rows(frame, fit_ids, val_ids, fold)\n    columns = json.loads(DEF.read_text())["context_columns"]\n    query = np.r_[vi, hi, te]\n    p, meta = eng.predict(frame, ci, query, y, columns, SEED + fold, n_estimators=8, column_cap=500, chunk=128,\n                          memory_fraction=.70)\n    part = np.r_[["val"] * len(vi), ["holdout"] * len(hi), ["test"] * len(te)]\n    out = pd.DataFrame({"application_id": frame.application_id.iloc[query].to_numpy(), "part": part, "kumo": p})\n    atomic_parquet(out, OUT / f"f{fold}/kumo{sfx()}.parquet")\n    meta.pop("context_rows", None)\n    atomic_json(meta, OUT / f"f{fold}/kumo{sfx()}_meta.json")\n    print("kumo", fold, "peak GiB", meta.get("max_allocated_gib"), flush=True)\n\n\ndef stage_post(fold):\n    import hybrid_kumo as hk\n    import research_context_correction054 as cc\n\n    frame, fit_ids, val_ids = fold_frame(fold)\n    y, ti, ci, hi, vi, te = split_rows(frame, fit_ids, val_ids, fold)\n    frame = frame.assign(target=np.where(np.isin(np.arange(len(frame)), ti), y, np.nan))\n    k = pd.read_parquet(OUT / f"f{fold}/kumo{sfx()}.parquet").set_index("application_id").kumo\n    kh, kv, kt = (k.loc[frame.application_id.iloc[ix]].to_numpy() for ix in (hi, vi, te))\n    d = json.loads(DEF.read_text())\n    spec = {**SPEC, "correction_columns": d["correction_columns"]}\n    bundle = cc.fit_correction(spec, frame, hi, kh)\n    cv_, _ = cc.predict_correction(bundle, frame.iloc[vi], kv)\n    ct_, _ = cc.predict_correction(bundle, frame.iloc[te], kt)\n    spline_spec, _ = hk.specifications()\n    sb = hk.fit_spline(spline_spec, frame, ti)\n    sv, st = hk.predict_spline(sb, frame.iloc[vi]), hk.predict_spline(sb, frame.iloc[te])\n    rows = []\n    for part, ix, kk, kc, ss in (("val", vi, kv, cv_, sv), ("test", te, kt, ct_, st)):\n        rows.append(pd.DataFrame({"application_id": frame.application_id.iloc[ix].to_numpy(), "part": part,\n                                  "K5": kk, "K5o": kc, "S": ss, "R0": .75 * kk + .25 * ss, "R": .75 * kc + .25 * ss}))\n    atomic_parquet(pd.concat(rows, ignore_index=True), OUT / f"f{fold}/post{sfx()}.parquet")\n    print("post", fold, "correction iters", bundle["iterations"], flush=True)\n\n\ndef stage_eval():\n    lab = labels()\n    folds = v.outer(lab.target.to_numpy(), 42)\n    names = ["K5", "K5o", "S", "R0", "R"]\n    oof = pd.DataFrame({"application_id": lab.application_id})\n    fold_ks = {n: [] for n in names}\n    for f, (_, va) in enumerate(folds):\n        p = pd.read_parquet(OUT / f"f{f}/post{sfx()}.parquet").query("part == \'val\'").set_index("application_id")\n        ids = lab.application_id.iloc[va]\n        for n in names:\n            oof.loc[va, n] = p.loc[ids, n].to_numpy()\n            fold_ks[n].append(v.ks(lab.target.iloc[va].to_numpy(), oof.loc[va, n].to_numpy()))\n    y = lab.target.to_numpy()\n    rep = {n: {"pooled_ks": v.ks(y, oof[n].to_numpy()), "auc": float(roc_auc_score(y, oof[n])),\n               "fold_ks": fold_ks[n], "mean_fold_ks": float(np.mean(fold_ks[n]))} for n in names}\n    for n, r in rep.items():\n        print(f"{n:4s} pooled {r[\'pooled_ks\']:.5f} auc {r[\'auc\']:.5f} mean-fold {r[\'mean_fold_ks\']:.5f}",\n              [round(x, 4) for x in r["fold_ks"]])\n    atomic_parquet(oof.assign(target=y), OUT / f"oof{sfx()}.parquet")\n    test = pd.concat([pd.read_parquet(OUT / f"f{f}/post{sfx()}.parquet").query("part == \'test\'") for f in range(5)])\n    atomic_parquet(test.groupby("application_id")[names].mean().reset_index(), OUT / f"test{sfx()}.parquet")\n    atomic_json(rep, ROOT / f"eda/out/251_teammate_s42{sfx()}.json")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--stage", required=True, choices=["reverse", "kumo", "post", "eval"])\n    ap.add_argument("--fold", type=int)\n    ap.add_argument("--seed", type=int, default=2026)\n    a = ap.parse_args()\n    SEED = a.seed\n    {"reverse": stage_reverse, "kumo": stage_kumo, "post": stage_post}.get(a.stage, lambda f: stage_eval())(a.fold)\n', 'eda/254_kumo_dsc_features.py': '"""Third Kumo view: DSC0152 feature engineering (598 cols + per-fold factor features), K64 cap300@28k view, seed 42 folds.\n\nProtocol: artifacts/optimization/continuation_25okt/protocol_254_kumo_dsc.json (locked before scores).\nUsage: cd eda && ../.venv/bin/python 254_kumo_dsc_features.py --stage {kumo --fold k, eval}\n"""\nimport argparse\nimport importlib\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nimport valid2 as v\nfrom common import ROOT\n\nt251 = importlib.import_module("251_teammate_recipe_s42")\nOUT = t251.OUT\nCACHE = ROOT / "scripts/dsc0152/agas/shared/cache"\nSEED = 2026  # --seed 2027 = second seed (protocol 257)\n\n\ndef sfx():\n    return "" if SEED == 2026 else f"_s{SEED}"\n\n\ndef dsc_frame():\n    """features_cache + extra_cache exactly as train_submission_v66.load_frames, all 100k rows."""\n    f = pd.read_pickle(CACHE / "features_cache.pkl")\n    ex = pd.read_pickle(CACHE / "extra_cache.pkl")\n    add = [c for c in ex.columns if c != "application_id" and c not in f.columns]\n    f = f.merge(ex[["application_id"] + add], on="application_id", how="left")\n    cols = [c for c in f.columns if c not in ("application_id", "customer_id", "target")]\n    assert len(cols) == 598, len(cols)\n    return f.drop(columns=["target"]), cols\n\n\ndef stage_kumo(fold):\n    import strict_kumo_engine as eng\n\n    frame, cols = dsc_frame()\n    fac = pd.read_parquet(OUT / f"f{fold}/factor.parquet")\n    frame = frame.merge(fac, on="application_id", how="left", validate="one_to_one")\n    lab = t251.labels()\n    tr, va = v.outer(lab.target.to_numpy(), 42)[fold]\n    pos = pd.Series(np.arange(len(frame)), index=frame.application_id)\n    fit = pos.loc[lab.application_id.iloc[tr]].to_numpy()\n    val = pos.loc[lab.application_id.iloc[va]].to_numpy()\n    test = np.setdiff1d(np.arange(len(frame)), np.r_[fit, val])\n    assert len(test) == 30000\n    y = np.zeros(len(frame), dtype=int)\n    y[fit] = lab.target.iloc[tr].to_numpy()\n    for c in frame.columns:  # engine accepts numeric/object/category only\n        if frame[c].dtype == bool:\n            frame[c] = frame[c].astype(float)\n    columns = cols + list(fac.columns[1:])\n    query = np.r_[val, test]\n    p, meta = eng.predict(frame, fit, query, y, columns, SEED + fold, n_estimators=8, max_context=28672,\n                          column_cap=300, chunk=128, memory_fraction=.85)\n    part = np.r_[["val"] * len(val), ["test"] * len(test)]\n    t251.atomic_parquet(pd.DataFrame({"application_id": frame.application_id.iloc[query].to_numpy(), "part": part,\n                                      "KD": p}), OUT / f"f{fold}/kumo_dsc{sfx()}.parquet")\n    meta.pop("context_rows", None)\n    t251.atomic_json(meta, OUT / f"f{fold}/kumo_dsc{sfx()}_meta.json")\n    print("kumo_dsc", fold, "peak GiB", meta.get("max_allocated_gib"), flush=True)\n\n\ndef stage_eval():\n    lab = t251.labels()\n    y = lab.target.to_numpy()\n    oof, fks = np.full(len(y), np.nan), []\n    for f, (_, va) in enumerate(v.outer(y, 42)):\n        p = pd.read_parquet(OUT / f"f{f}/kumo_dsc{sfx()}.parquet").query("part == \'val\'").set_index("application_id")\n        oof[va] = p.loc[lab.application_id.iloc[va], "KD"].to_numpy()\n        fks.append(v.ks(y[va], oof[va]))\n    rep = {"pooled_ks": v.ks(y, oof), "auc": float(roc_auc_score(y, oof)), "fold_ks": fks}\n    print(f"KD pooled {rep[\'pooled_ks\']:.5f} auc {rep[\'auc\']:.5f}", [round(x, 4) for x in fks], flush=True)\n    t251.atomic_parquet(pd.DataFrame({"application_id": lab.application_id, "target": y, "KD": oof}),\n                        OUT / f"oof_kd{sfx()}.parquet")\n    test = pd.concat([pd.read_parquet(OUT / f"f{f}/kumo_dsc{sfx()}.parquet").query("part == \'test\'") for f in range(5)])\n    t251.atomic_parquet(test.groupby("application_id")[["KD"]].mean().reset_index(), OUT / f"test_kd{sfx()}.parquet")\n    t251.atomic_json(rep, ROOT / f"eda/out/254_kumo_dsc{sfx()}.json")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--stage", required=True, choices=["kumo", "eval"])\n    ap.add_argument("--fold", type=int)\n    ap.add_argument("--seed", type=int, default=2026)\n    a = ap.parse_args()\n    SEED = a.seed\n    stage_kumo(a.fold) if a.stage == "kumo" else stage_eval()\n', 'eda/valid2.py': '"""validation_v2: one protocol for local screening (one fold) and full Kaggle runs.\n\nOuter: StratifiedKFold(5, shuffle, seed) per seed; the outer validation fold is only scored, never used for any decision.\nInner: a stratified 1/7 split of the outer training fold picks the iteration / epoch (early stopping). The model is then\nrefitted on the whole outer training fold with that fixed budget (GBDT: best x REFIT, TabM: best epoch).\nEvery run writes OOF and test predictions with application_id. Blend weights are chosen later, cross-fitted over the\nouter folds (eda/blend2.py), never on the rows they are scored on."""\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport polars as pl\nfrom scipy.stats import ks_2samp\nfrom sklearn.model_selection import StratifiedKFold\n\nID_COLS = ["application_id", "customer_id", "decision_datetime", "target", "is_train"]\nCAT_COLS = ["loan_purpose", "region", "employment_type", "housing_status"]\nN_FOLDS, INNER, REFIT = 5, 7, 1.15\nSEEDS = [42, 7, 2026]\nN_ROUNDS, EARLY_STOP = 10000, 200\nTHREADS = os.cpu_count() or 4\n\nLGB = dict(objective="binary", metric="auc", learning_rate=0.02, num_leaves=31, min_child_samples=100,\n           feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, n_jobs=THREADS)\nLGB_S = dict(LGB, num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4, lambda_l2=10.0)\nXGB = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.02, max_depth=5, min_child_weight=5,\n           subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, tree_method="hist", device="cuda")\nXGB_S = dict(XGB, max_depth=3, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)\nCAT = dict(learning_rate=0.03, depth=6, l2_leaf_reg=5, loss_function="Logloss", eval_metric="AUC",\n           task_type="GPU", verbose=0)\nCAT_S = dict(CAT, depth=4, l2_leaf_reg=10)\nLGB_X = dict(LGB, extra_trees=True)\nXGB_D8 = dict(XGB, max_depth=8, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)\nCAT_D8 = dict(CAT, depth=8, l2_leaf_reg=10)\nTABM = dict(lr=2e-3, wd=3e-4, bins=48, batch=512, epochs=60, patience=8)\n\n\ndef ks(y, p):\n    return float(ks_2samp(p[y == 1], p[y == 0]).statistic)\n\n\ndef outer(y, seed):\n    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))\n\n\ndef inner(y_tr, seed, fold):\n    """(fit, stop) positions inside the outer training fold."""\n    return next(StratifiedKFold(INNER, shuffle=True, random_state=1000 * seed + fold).split(np.zeros(len(y_tr)), y_tr))\n\n\ndef frames(features: pl.DataFrame):\n    """features (build_all output) -> X, Xt (pandas; categoricals typed with train categories), y, train ids, test ids."""\n    f = features.sort("application_id")\n    tr, te = f.filter(pl.col("is_train") == 1), f.filter(pl.col("is_train") == 0)\n    cols = sorted(c for c in f.columns if c not in ID_COLS)\n    X, Xt = tr.select(cols).to_pandas(), te.select(cols).to_pandas()\n    for c in cols:\n        if c in CAT_COLS or f[c].dtype == pl.String:\n            cats = sorted(X[c].dropna().unique())\n            X[c], Xt[c] = pd.Categorical(X[c], categories=cats), pd.Categorical(Xt[c], categories=cats)\n        else:\n            X[c] = X[c].astype(float).replace([np.inf, -np.inf], np.nan)\n            Xt[c] = Xt[c].astype(float).replace([np.inf, -np.inf], np.nan)\n    return X, Xt, tr["target"].to_numpy().astype(int), tr["application_id"], te["application_id"]\n\n\n# --- model fits: fit(X, y, tr, fit, stop, rest, seed) -> list of predictions for each frame in rest ---------------\n\ndef lgb_fit(params):\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]])\n        d_st = lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)\n        m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[d_st], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n        n = math.ceil(m.best_iteration * REFIT)\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), n)\n        return [m.predict(r) for r in rest], n\n    return fit\n\n\nLGB_GRID = [  # diverse configurations for nested selection (lgb_nested); chosen per outer fold on inner-stop KS only\n    {}, dict(num_leaves=15, min_child_samples=200), dict(num_leaves=63, min_child_samples=100),\n    dict(num_leaves=127, min_child_samples=300, lambda_l2=20.0), dict(num_leaves=31, feature_fraction=0.3),\n    dict(num_leaves=31, feature_fraction=0.8), dict(num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4,\n                                                  lambda_l2=10.0), dict(num_leaves=31, min_child_samples=500),\n    dict(num_leaves=255, min_child_samples=500, learning_rate=0.01, lambda_l2=30.0), dict(num_leaves=31, extra_trees=True),\n]\n\n\ndef lgb_nested_fit(grid=LGB_GRID):\n    """Clean nested model selection: every config early-stops on the inner stop split; the config with the best\n    inner-stop KS is refitted on the whole training fold (budget x REFIT). No outer-fold label is used."""\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        best = None\n        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]], free_raw_data=False)\n        for j, g in enumerate(grid):\n            p = dict(LGB, **g, seed=seed)\n            m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)],\n                          callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n            k = ks(y[tr[st]], m.predict(X.iloc[tr[st]], num_iteration=m.best_iteration))\n            if best is None or k > best[0]:\n                best = (k, j, p, m.best_iteration)\n        k, j, p, it = best\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), math.ceil(it * REFIT))\n        print(f"lgb_nested: config {j} {grid[j]} inner-stop KS {k:.4f} rounds {it}", flush=True)\n        return [m.predict(r) for r in rest], j\n    return fit\n\n\ndef xgb_fit(params):\n    import xgboost as xgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = xgb.DMatrix(X.iloc[tr[fi]], y[tr[fi]], enable_categorical=True)\n        d_st = xgb.DMatrix(X.iloc[tr[st]], y[tr[st]], enable_categorical=True)\n        m = xgb.train(p, d_fit, N_ROUNDS, evals=[(d_st, "st")], early_stopping_rounds=EARLY_STOP, verbose_eval=False)\n        n = math.ceil((m.best_iteration + 1) * REFIT)\n        m = xgb.train(p, xgb.DMatrix(X.iloc[tr], y[tr], enable_categorical=True), n)\n        return [m.predict(xgb.DMatrix(r, enable_categorical=True)) for r in rest], n\n    return fit\n\n\ndef cat_fit(params):\n    from catboost import CatBoostClassifier, Pool\n\n    def s(df):\n        out = df.copy()\n        for c in CAT_COLS:\n            out[c] = out[c].astype(str)\n        return out\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cats = [c for c in CAT_COLS if c in X.columns]\n        m = CatBoostClassifier(**params, iterations=N_ROUNDS, od_type="Iter", od_wait=EARLY_STOP, use_best_model=True,\n                               random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr[fi]]), y[tr[fi]], cat_features=cats), eval_set=Pool(s(X.iloc[tr[st]]), y[tr[st]], cat_features=cats))\n        n = math.ceil((m.get_best_iteration() + 1) * REFIT)\n        m = CatBoostClassifier(**params, iterations=n, random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr]), y[tr], cat_features=cats))\n        return [m.predict_proba(s(r))[:, 1] for r in rest], n\n    return fit\n\n\ndef tabm_fit(cfg=TABM, device="cuda"):\n    import torch\n    import tabm\n    import rtdl_num_embeddings\n    from sklearn.metrics import roc_auc_score\n    from nn_prep import nn_prep\n\n    def arrays(X, rows_list):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        return [X.iloc[r][nums].to_numpy(dtype=float) for r in rows_list]\n\n    def codes(df):\n        return np.column_stack([df[c].cat.codes.to_numpy() + 1 for c in CAT_COLS]).astype(np.int64)\n\n    def train(xn, xc, yv, cards, n_epochs, seed, watch=None):\n        torch.manual_seed(seed)\n        dev = torch.device(device)\n        xn_t, xc_t = torch.as_tensor(xn, device=dev), torch.as_tensor(xc, device=dev)\n        yt = torch.as_tensor(yv, dtype=torch.float32, device=dev)\n        emb = rtdl_num_embeddings.PiecewiseLinearEmbeddings(\n            rtdl_num_embeddings.compute_bins(xn_t, n_bins=cfg["bins"]), d_embedding=16, activation=False, version="B")\n        model = tabm.TabM.make(n_num_features=xn.shape[1], cat_cardinalities=cards, d_out=1, num_embeddings=emb).to(dev)\n        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])\n\n        def predict(a, c):\n            model.eval()\n            a, c = torch.as_tensor(a, device=dev), torch.as_tensor(c, device=dev)\n            with torch.no_grad():\n                return torch.cat([torch.sigmoid(model(a[i:i + 1024], c[i:i + 1024]).squeeze(-1).float()).mean(1)\n                                  for i in range(0, len(a), 1024)]).cpu().numpy()\n\n        best, best_ep, wait = -1.0, n_epochs, 0\n        for ep in range(1, n_epochs + 1):\n            model.train()\n            for idx in torch.randperm(len(xn_t), device=dev).split(cfg["batch"]):\n                out = model(xn_t[idx], xc_t[idx]).squeeze(-1)\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(out, yt[idx].unsqueeze(1).expand_as(out))\n                opt.zero_grad()\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                opt.step()\n            if watch is not None:\n                auc = roc_auc_score(watch[2], predict(watch[0], watch[1]))\n                best, best_ep, wait = (auc, ep, 0) if auc > best else (best, best_ep, wait + 1)\n                if wait >= cfg["patience"]:\n                    break\n        return model, predict, best_ep\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cards = [len(X[c].cat.categories) + 1 for c in CAT_COLS]\n        a_fit, a_st = arrays(X, [tr[fi], tr[st]])\n        x_fit, x_st, _ = nn_prep(a_fit, a_st)\n        _, _, ep = train(x_fit, codes(X.iloc[tr[fi]]), y[tr[fi]], cards, cfg["epochs"], seed,\n                         watch=(x_st, codes(X.iloc[tr[st]]), y[tr[st]]))\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a_tr = X.iloc[tr][nums].to_numpy(dtype=float)\n        a_rest = np.vstack([r[nums].to_numpy(dtype=float) for r in rest])\n        x_tr, x_rest, _ = nn_prep(a_tr, a_rest)\n        model, predict, _ = train(x_tr, codes(X.iloc[tr]), y[tr], cards, ep, seed)\n        p = predict(x_rest, np.vstack([codes(r) for r in rest]))\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), ep\n    return fit\n\n\ndef ks_nan(y, x):\n    x = np.asarray(x, float)\n    if np.isnan(x).all():\n        return 0.0\n    lo, hi = np.nanmin(x) - 1, np.nanmax(x) + 1\n    return max(ks(y, np.where(np.isnan(x), lo, x)), ks(y, np.where(np.isnan(x), hi, x)))\n\n\ndef fm_select(X, y, tr, n_num=100, rand=0, n_top=20):\n    """v8 recipe: numeric columns ranked by univariate KS on the training fold, greedy |Spearman| < 0.95, top n_num,\n    plus the 4 categoricals. rand > 0 (feature bagging): the n_top best kept, the other n_num - n_top drawn at random\n    (seed rand) from every column that passes the Spearman filter. rand < 0: rank by LGB gain (seed -rand) instead."""\n    nums = [c for c in X.columns if c not in CAT_COLS]\n    sub = X.iloc[tr]\n    if rand < 0:  # rank by LGB split gain on the training fold (conditional usefulness) instead of univariate KS\n        import lightgbm as lgb\n        m = lgb.train(dict(LGB, seed=-rand), lgb.Dataset(sub[nums], y[tr]), 400)\n        gain = dict(zip(nums, m.feature_importance("gain")))\n        ranked = sorted(nums, key=lambda c: -gain[c])[:2 * n_num]\n        rand = 0\n    else:\n        ranked = sorted(nums, key=lambda c: -ks_nan(y[tr], sub[c].to_numpy(dtype=float)))[:None if rand else 2 * n_num]\n    corr = sub[ranked].rank().corr().abs().fillna(0.0)\n    keep = []\n    for c in ranked:\n        if all(corr.loc[c, k] < 0.95 for k in keep):\n            keep.append(c)\n        if len(keep) == n_num and not rand:\n            break\n    if rand:\n        rest = list(np.random.default_rng(rand).permutation(keep[n_top:]))\n        keep = keep[:n_top] + rest[:n_num - n_top]\n    return keep + CAT_COLS\n\n\ndef icl_fit(make, n_num=100, max_context=60000, chunk=10000, as_codes=False, rand=0):\n    """In-context learners: no early stopping (the inner split is unused). make(seed, cat_idx) -> sklearn-like model."""\n    def fit(X, y, tr, fi, st, rest, seed):\n        cols = fm_select(X, y, tr, n_num, rand)\n        ctx = tr if len(tr) <= max_context else np.random.default_rng(seed).choice(tr, max_context, replace=False)\n\n        def frame(d):\n            d = d[cols].reset_index(drop=True)\n            if as_codes:\n                d = d.copy()\n                for c in CAT_COLS:\n                    d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)\n                return d.to_numpy(dtype=np.float32)\n            return d\n        m = make(seed, [cols.index(c) for c in CAT_COLS])\n        m.fit(frame(X.iloc[ctx]), y[ctx])\n        preds = [np.concatenate([np.asarray(m.predict_proba(frame(r.iloc[i:i + chunk])))[:, 1]\n                                 for i in range(0, len(r), chunk)]) for r in rest]\n        import torch\n        del m\n        gc.collect()\n        torch.cuda.empty_cache()\n        return preds, len(cols)\n    return fit\n\n\ndef seg_fit(base):\n    """Fit `base` separately on thin-file (has_history=0) and history rows: each segment gets its own context and its\n    own fm_select, so thin-file queries are not given ~half all-NaN history columns."""\n    def fit(X, y, tr, fi, st, rest, seed):\n        h = X["has_history"].to_numpy()\n        out = [np.zeros(len(r)) for r in rest]\n        for s in (0, 1):\n            ts = tr[h[tr] == s]\n            masks = [r["has_history"].to_numpy() == s for r in rest]\n            preds, n = base(X, y, ts, fi, st, [r[m] for r, m in zip(rest, masks)], seed)\n            for o, p_, m in zip(out, preds, masks):\n                o[m] = p_\n        return out, n\n    return fit\n\n\ndef stack_fit(base, k=5):\n    """Stacked ICL: `base` gets one extra column gb_stk = LGB raw score. Context rows (training fold) carry nested k-fold\n    OOF scores from LGB fits inside the training fold; query rows carry the mean score of those same k fits (same\n    training size as the context scores: a refit on the whole fold gave sharper query scores, Causilo fold 0 -0.0027).\n    LGB budget comes from the inner stop split, so no outer-fold label is touched."""\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(LGB, seed=seed)\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]]), N_ROUNDS,\n                      valid_sets=[lgb.Dataset(X.iloc[tr[st]], y[tr[st]])],\n                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n        n = math.ceil(m.best_iteration * REFIT)\n        g, gq = np.full(len(X), np.nan), [np.zeros(len(r)) for r in rest]\n        for a, b in StratifiedKFold(k, shuffle=True, random_state=seed + 17).split(tr, y[tr]):\n            m = lgb.train(p, lgb.Dataset(X.iloc[tr[a]], y[tr[a]]), n)\n            g[tr[b]] = m.predict(X.iloc[tr[b]], raw_score=True)\n            for q, r in zip(gq, rest):\n                q += m.predict(r, raw_score=True) / k\n        print(f"stack_fit: gb_stk sd context {np.nanstd(g[tr]):.3f} query {np.std(gq[0]):.3f}", flush=True)\n        preds, _ = base(X.assign(gb_stk=g), y, tr, fi, st, [r.assign(gb_stk=q) for r, q in zip(rest, gq)], seed)\n        return preds, n\n    return fit\n\n\ndef stack_lr_fit(base, k=5):\n    """Supervised oblique direction for trees: column lr_stk = nested logit of lr_q (rank-gauss LR). Training-fold rows\n    carry k-fold OOF lr_q scores fitted inside the training fold; query rows the mean of those k fits. No outer label."""\n    lr = lr_q_fit()\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        g, gq = np.full(len(X), np.nan), [np.zeros(len(r)) for r in rest]\n        for a, b in StratifiedKFold(k, shuffle=True, random_state=seed + 23).split(tr, y[tr]):\n            sub = tr[a]\n            f2, s2 = inner(y[sub], seed, 99)\n            preds, _ = lr(X, y, sub, f2, s2, [X.iloc[tr[b]], *rest], seed)\n            g[tr[b]] = preds[0]\n            for q, p_ in zip(gq, preds[1:]):\n                q += p_ / k\n        lg = lambda p_: np.log(np.clip(p_, 1e-6, 1 - 1e-6) / (1 - np.clip(p_, 1e-6, 1 - 1e-6)))  # noqa: E731\n        return base(X.assign(lr_stk=lg(g)), y, tr, fi, st, [r.assign(lr_stk=lg(q)) for r, q in zip(rest, gq)], seed)\n    return fit\n\n\ndef pca_fit(base, k=20):\n    """Rotation features for axis-aligned trees: rank-gauss (nn_prep, fitted on the training fold) -> PCA fitted on the\n    training fold -> k components appended as pca_0..k-1. Unsupervised, so no label enters."""\n    from nn_prep import nn_prep\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        A_tr, A_all, _ = nn_prep(X.iloc[tr][nums].to_numpy(float), X[nums].to_numpy(float))\n        mu = A_tr.mean(0)\n        _, _, vt = np.linalg.svd(A_tr - mu, full_matrices=False)\n        W = vt[:k].T\n        add = lambda d: d.assign(**{f"pca_{i}": c for i, c in enumerate(((nn_prep(X.iloc[tr][nums].to_numpy(float), d[nums].to_numpy(float))[1] - mu) @ W).T)})  # noqa: E731\n        Xa = X.assign(**{f"pca_{i}": c for i, c in enumerate(((A_all - mu) @ W).T)})\n        return base(Xa, y, tr, fi, st, [add(r) for r in rest], seed)\n    return fit\n\n\ndef causilo_fit(n_num=100, n_est=4, max_context=60000, device="cuda", rand=0):\n    from causilo import CausiloClassifier\n    return icl_fit(lambda seed, ci: CausiloClassifier(n_estimators=n_est, random_state=seed, device=device),\n                   n_num, max_context, rand=rand)\n\n\ndef tabpfn_fit(n_num=100, n_est=4, max_context=60000, device="cuda"):\n    from tabpfn import TabPFNClassifier\n    return icl_fit(lambda seed, ci: TabPFNClassifier(n_estimators=n_est, random_state=seed, device=device,\n                                                     ignore_pretraining_limits=True, categorical_features_indices=ci,\n                                                     fit_mode="fit_with_cache"),\n                   n_num, max_context, chunk=5000, as_codes=True)\n\n\ndef tabdpt_fit(n_num=100, context=4096, n_ens=8, device="cuda"):\n    """TabDPT retrieves `context` nearest training rows per query (its own context reduction)."""\n    from tabdpt import TabDPTClassifier\n\n    class M:\n        def __init__(self, seed):\n            self.m, self.seed = TabDPTClassifier(device=device, context_reduction="retrieval", missing_indicators=True,\n                                                 compile=False, verbose=False), seed\n\n        def fit(self, Xa, ya):\n            self.m.fit(Xa, ya)\n\n        def predict_proba(self, Xa):\n            return np.mean([self.m.predict_proba(Xa, context_size=context, batch_size=64, seed=self.seed + k) for k in range(n_ens)], axis=0)\n    return icl_fit(lambda seed, ci: M(seed), n_num, 10 ** 9, chunk=2048, as_codes=True)\n\n\ndef kumo_fit(size="large", n_num=100, max_context=20000, n_est=4, chunk=1000, n_bag=1, g_off=0, rand=0):\n    """NVIDIA Kumo-Tabular (structured-data-models) in-context learner on fm_select features (categoricals as integer\n    codes, numeric-only model). Context = stratified subsample of `max_context` training-fold rows; no early stopping."""\n    import torch\n    import sdm\n\n    def frame(d, cols):\n        d = d[cols].copy()\n        for c in CAT_COLS:\n            d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)\n        return d.astype(float).reset_index(drop=True)\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        """n_bag context draws (stratified, independent) averaged: one 20k draw moves fold KS by up to ~0.01."""\n        cols = fm_select(X, y, tr, n_num, rand)\n        dev = torch.device("cuda")\n        model = sdm.models.KumoTabular(task="classification", size=size, device=dev)\n        q = pd.concat([frame(r, cols) for r in rest], ignore_index=True)\n        p = np.mean([one(model, X, y, tr, cols, q, seed * 100 + b, dev, g_off) for b in range(n_bag)], axis=0)\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), len(tr) if len(tr) <= max_context else max_context\n\n    def one(model, X, y, tr, cols, q, seed, dev, g_off=0):\n        rng = np.random.default_rng(seed)\n        ctx = tr if len(tr) <= max_context else np.sort(np.concatenate(\n            [rng.choice(tr[y[tr] == c], int(round(max_context * (y[tr] == c).mean())), replace=False) for c in (0, 1)]))\n        cdf = frame(X.iloc[ctx], cols)\n        full = pd.concat([cdf.assign(target=y[ctx].astype(str)), q.assign(target="0")], ignore_index=True)\n        stypes = sdm.infer_stypes(full, overrides={"target": "categorical", **{c: "numerical" for c in cols}})\n        t = sdm.TableTensor.from_pandas(df=full, stypes=stypes, device=dev)\n        n = len(cdf)\n        xc, yc = t[:n].drop_columns("target"), t[:n, "target"]\n        g = torch.Generator(device=dev).manual_seed(seed + 1000 * g_off)  # g_off: independent estimator draw, same context\n        out = []\n        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):\n            for i in range(n, len(full), chunk):\n                r = model(x_context=xc, y_context=yc, x_query=t[i:min(i + chunk, len(full))].drop_columns("target"),\n                          num_estimators=n_est, generator=g)\n                out.append(r.to_pandas()["1"].to_numpy())\n        del t\n        return np.concatenate(out)\n    return fit\n\n\ndef _torch_faiss_gpu():\n    """shortcut: CPU-only faiss is installed but pytabkit TabR asks for faiss.GpuIndexFlatL2; an exact torch L2 index\n    (cdist + topk on the GPU) stands in. Upgrade to faiss-gpu if TabR becomes part of a recipe."""\n    import faiss\n    import torch\n\n    class Index:\n        def __init__(self, *a):\n            self.x = None\n\n        def reset(self):\n            self.x = None\n\n        def add(self, x):\n            self.x = x\n\n        def search(self, q, k):\n            d = torch.cdist(q.float(), self.x.float()) ** 2\n            dist, idx = d.topk(k, dim=1, largest=False)\n            return dist, idx\n\n    faiss.GpuIndexFlatConfig = type("Cfg", (), {"device": 0})\n    faiss.StandardGpuResources = lambda: None\n    faiss.GpuIndexFlatL2 = Index\n\n\ndef realmlp_fit(n_ens=1, model="RealMLP_TD_Classifier", **kw):\n    """pytabkit RealMLP-TD (tuned defaults; `model` = another pytabkit class, e.g. TabR_S_D_Classifier). Checkpoint\n    selection only on the inner stop split (X_val), then the selected network predicts the outer fold and test (no\n    refit: the schedule is tied to its epoch budget)."""\n    import pytabkit\n    cls = getattr(pytabkit, model)\n    if model.startswith("RealMLP"):\n        kw |= dict(n_ens=n_ens, val_metric_name="1-auc_ovr")  # TabR only has its default val metric\n    else:\n        _torch_faiss_gpu()\n\n    def prep(X, rows):\n        """Median fill fitted on `rows`, one indicator per distinct missingness pattern (thin-file blocks)."""\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        med = X.iloc[rows][nums].median()\n        miss = X.iloc[rows][nums].isna()\n        groups, seen = [], set()\n        for c in nums:\n            key = np.packbits(miss[c].to_numpy()).tobytes()\n            if miss[c].any() and key not in seen:\n                seen.add(key)\n                groups.append(c)\n\n        def tf(d):\n            out = d[nums].fillna(med).fillna(0.0)\n            for c in groups:\n                out[f"na_{c}"] = d[c].isna().astype(float)\n            for c in CAT_COLS:\n                out[c] = d[c]\n            return out\n        return tf\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        tf = prep(X, tr[fi])\n        X = tf(X)\n        rest = [tf(r) for r in rest]\n        cats = CAT_COLS\n        if not model.startswith("RealMLP"):  # pytabkit\'s cat encoder fails on an empty group under sklearn 1.9\n            X = pd.get_dummies(X, columns=CAT_COLS, dtype=float)\n            rest = [pd.get_dummies(r, columns=CAT_COLS, dtype=float) for r in rest]\n            cats = []\n        m = cls(device="cuda", random_state=seed, verbosity=0, **kw)\n        m.fit(X.iloc[tr[fi]], y[tr[fi]], X_val=X.iloc[tr[st]], y_val=y[tr[st]], cat_col_names=cats)\n        return [m.predict_proba(r)[:, 1] for r in rest], 0\n    return fit\n\n\ndef woe_lr_fit(n_bins=20, alpha=20.0, Cs=(0.003, 0.01, 0.03, 0.1, 0.3)):\n    """Credit scorecard: per feature, bins fitted on the training rows (NaN and exact zero get their own bins, else\n    quantile bins of the rest; categoricals by level), smoothed WOE = log odds of the bin shrunk to the prior with\n    `alpha` pseudo-rows, then L2 logistic regression on the standardised WOE columns. C is picked on the inner stop\n    split, then the LR is refitted on the whole training fold (binning refitted there too)."""\n    from sklearn.linear_model import LogisticRegression\n\n    def encoder(X, rows, yv):\n        prior = yv.mean()\n        lo = np.log(prior / (1 - prior))\n        specs = []\n        for c in X.columns:\n            if c in CAT_COLS:\n                codes = X[c].cat.codes.to_numpy()\n                specs.append((c, "cat", None))\n                b = codes[rows]\n            else:\n                x = X[c].to_numpy(dtype=float)[rows]\n                nz = np.isfinite(x) & (x != 0)\n                edges = np.unique(np.quantile(x[nz], np.linspace(0, 1, n_bins + 1))[1:-1]) if nz.sum() > n_bins else np.array([])\n                specs.append((c, "num", edges))\n                b = bins_of(x, edges)\n            keys, inv = np.unique(b, return_inverse=True)\n            pos = np.bincount(inv, weights=yv, minlength=len(keys))\n            n = np.bincount(inv, minlength=len(keys))\n            rate = (pos + alpha * prior) / (n + alpha)\n            specs[-1] = specs[-1] + (dict(zip(keys.tolist(), (np.log(rate / (1 - rate)) - lo).tolist())),)\n        return specs\n\n    def bins_of(x, edges):\n        out = np.where(np.isnan(x), -2, np.where(x == 0, -1, np.searchsorted(edges, x, side="right")))\n        return out\n\n    def transform(specs, D):\n        cols = []\n        for c, kind, edges, table in specs:\n            b = D[c].cat.codes.to_numpy() if kind == "cat" else bins_of(D[c].to_numpy(dtype=float), edges)\n            cols.append(np.array([table.get(int(k), 0.0) for k in b]) if len(table) < 50 else\n                        np.vectorize(lambda k: table.get(int(k), 0.0))(b))\n        return np.column_stack(cols)\n\n    def fit_lr(A, yv, C, seed):\n        mu, sd = A.mean(0), A.std(0) + 1e-9\n        m = LogisticRegression(C=C, max_iter=2000, random_state=seed).fit((A - mu) / sd, yv)\n        return lambda B: m.decision_function((B - mu) / sd)\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        sp = encoder(X, tr[fi], y[tr[fi]])\n        A, S = transform(sp, X.iloc[tr[fi]]), transform(sp, X.iloc[tr[st]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], fit_lr(A, y[tr[fi]], C, seed)(S)))\n        sp = encoder(X, tr, y[tr])\n        f = fit_lr(transform(sp, X.iloc[tr]), y[tr], best, seed)\n        return [1 / (1 + np.exp(-f(transform(sp, r)))) for r in rest], best\n    return fit\n\n\ndef lr_q_fit(Cs=(0.001, 0.003, 0.01, 0.03, 0.1, 1.0), onehot=True):\n    """Logistic regression on nn_prep inputs (mid-rank ECDF -> normal fitted on the training rows, NaN -> 0, one\n    indicator per missingness pattern) + one-hot categoricals. C picked on the inner stop split, refit on the fold."""\n    from sklearn.linear_model import LogisticRegression\n    from nn_prep import nn_prep\n\n    def mats(X, fit_rows, others):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a, b, _ = nn_prep(X.iloc[fit_rows][nums].to_numpy(dtype=float),\n                          np.vstack([o[nums].to_numpy(dtype=float) for o in others]))\n        def oh(d):\n            return np.column_stack([(d[c].cat.codes.to_numpy()[:, None] == np.arange(len(d[c].cat.categories))).astype(np.float32)\n                                    for c in CAT_COLS])\n        if onehot:\n            a = np.hstack([a, oh(X.iloc[fit_rows])])\n            b = np.hstack([b, np.vstack([oh(o) for o in others])])\n        return a, np.split(b, np.cumsum([len(o) for o in others])[:-1])\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        a, (s_,) = mats(X, tr[fi], [X.iloc[tr[st]]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], LogisticRegression(C=C, max_iter=3000).fit(a, y[tr[fi]]).decision_function(s_)))\n        a, outs = mats(X, tr, rest)\n        m = LogisticRegression(C=best, max_iter=3000).fit(a, y[tr])\n        return [m.predict_proba(o)[:, 1] for o in outs], best\n    return fit\n\n\ndef run(name, fit, X, Xt, y, ids, test_ids, out: Path, seeds=SEEDS, folds=None, deadline=None, test=True):\n    """Runs fit over seeds x outer folds. OOF = mean over seeds (NaN where no fold ran); test = mean over all fits.\n    Writes out/oof_<name>.parquet, test_<name>.parquet and log_<name>.csv. test=False (screening): no test predictions."""\n    out.mkdir(parents=True, exist_ok=True)\n    oof_sum, oof_n, test_sum, log = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(Xt)), []\n    for seed in seeds:\n        for k, (tr, va) in enumerate(outer(y, seed)):\n            if folds is not None and k not in folds:\n                continue\n            if deadline is not None and time.time() > deadline:\n                raise TimeoutError(f"{name}: deadline before seed {seed} fold {k}")\n            t0 = time.time()\n            fi, st = inner(y[tr], seed, k)\n            preds, budget = fit(X, y, tr, fi, st, [X.iloc[va], Xt] if test else [X.iloc[va]], seed)\n            pv, pt = preds[0], preds[1] if test else np.nan\n            oof_sum[va] += pv\n            oof_n[va] += 1\n            test_sum += pt\n            log.append(dict(model=name, seed=seed, fold=k, ks=ks(y[va], pv), budget=budget, sec=round(time.time() - t0, 1)))\n            print(log[-1], flush=True)\n            pd.DataFrame(log).to_csv(out / f"log_{name}.csv", index=False)\n            # written after every fold so an interrupted run keeps the folds it finished\n            oof = np.where(oof_n > 0, oof_sum / np.maximum(oof_n, 1), np.nan)\n            pl.DataFrame({"application_id": ids, "target": y, name: oof}).write_parquet(out / f"oof_{name}.parquet")\n            if test:\n                pl.DataFrame({"application_id": test_ids, name: test_sum / len(log)}).write_parquet(out / f"test_{name}.parquet")\n    m = oof_n > 0\n    print(f"{name}: OOF KS {ks(y[m], oof[m]):.5f} on {m.sum()} rows", flush=True)\n    return oof, test_sum / len(log), log\n\n\ndef manifest(out: Path, extra=None):\n    import importlib.metadata as md\n    pk = {}\n    for n in ["numpy", "pandas", "polars", "scikit-learn", "scipy", "lightgbm", "xgboost", "catboost", "torch", "tabm",\n              "rtdl_num_embeddings", "causilo"]:\n        try:\n            pk[n] = md.version(n)\n        except md.PackageNotFoundError:\n            pk[n] = None\n    cfg = dict(N_FOLDS=N_FOLDS, INNER=INNER, REFIT=REFIT, SEEDS=SEEDS, N_ROUNDS=N_ROUNDS, EARLY_STOP=EARLY_STOP,\n               LGB=LGB, LGB_S=LGB_S, XGB=XGB, XGB_S=XGB_S, CAT=CAT, CAT_S=CAT_S, TABM=TABM)\n    (out / "manifest.json").write_text(json.dumps(dict(packages=pk, config=cfg, **(extra or {})), indent=2, default=str))\n', 'eda/common.py': 'from pathlib import Path\nimport polars as pl\n\nROOT = Path(__file__).resolve().parent.parent\nDATA = ROOT / "data"\nPQ = ROOT / "data_pq"\nOUT = ROOT / "eda" / "out"\nOUT.mkdir(parents=True, exist_ok=True)\nPQ.mkdir(exist_ok=True)\n\nTABLES = ["train_applications", "test_applications", "customer_profiles", "previous_applications",\n          "repayment_history", "account_behavior", "bureau_observations", "ecommerce_transactions"]\n\n\ndef load(name: str) -> pl.DataFrame:\n    p = PQ / f"{name}.parquet"\n    if not p.exists():\n        pl.read_csv(DATA / f"{name}.csv", try_parse_dates=True, infer_schema_length=100000).write_parquet(p)\n    return pl.read_parquet(p)\n\n\ndef ks(y, s):\n    """KS exactly as the competition scorer (ks_2samp on bad vs good)."""\n    import numpy as np\n    from scipy.stats import ks_2samp\n    y, s = np.asarray(y), np.asarray(s, dtype=float)\n    m = ~np.isnan(s)\n    y, s = y[m], s[m]\n    if y.min() == y.max():\n        return 0.0\n    return float(ks_2samp(s[y == 1], s[y == 0]).statistic)\n\n\ndef ks_nan(y, s):\n    """Univariate KS where NaN is its own bin (placed at either extreme, best taken)."""\n    import numpy as np\n    s = np.asarray(s, dtype=float)\n    if np.isnan(s).all():\n        return 0.0\n    lo, hi = np.nanmin(s) - 1, np.nanmax(s) + 1\n    return max(ks(y, np.where(np.isnan(s), lo, s)), ks(y, np.where(np.isnan(s), hi, s)))\n\n\ndef psi(a, b, bins=10):\n    """Population stability index of b vs a using a\'s quantile bins (NaN = own bin)."""\n    import numpy as np\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    qs = np.unique(np.nanquantile(a, np.linspace(0, 1, bins + 1))) if (~np.isnan(a)).any() else np.array([0.0])\n    def dist(x):\n        idx = np.where(np.isnan(x), -1, np.clip(np.searchsorted(qs[1:-1], x, side="right"), 0, None))\n        cnt = np.array([(idx == k).sum() for k in range(-1, len(qs))], float) + 0.5\n        return cnt / cnt.sum()\n    pa, pb = dist(a), dist(b)\n    return float(((pb - pa) * np.log(pb / pa)).sum())\n\n\ndef features():\n    p = PQ / "features.parquet"\n    if not p.exists():\n        from features import build_all\n        build_all({t: load(t) for t in TABLES}).write_parquet(p)\n    return pl.read_parquet(p)\n\n\ndef analyze(df: pl.DataFrame, cols, tag: str, top_plot: int = 12):\n    """Prints KS / PSI / coverage table and saves decile bad-rate plots for the top features."""\n    import numpy as np\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    tr, te = df.filter(pl.col("is_train") == 1), df.filter(pl.col("is_train") == 0)\n    y = tr["target"].to_numpy()\n    rows = []\n    for c in cols:\n        if tr[c].dtype == pl.String:\n            rate = tr.group_by(c).agg(pl.col("target").mean()).rename({"target": "r"})\n            s_tr = tr.join(rate, on=c, how="left")["r"].to_numpy()\n            s_te = te.join(rate, on=c, how="left")["r"].to_numpy()\n        else:\n            s_tr, s_te = tr[c].cast(pl.Float64).to_numpy(), te[c].cast(pl.Float64).to_numpy()\n        rows.append((c, ks_nan(y, s_tr), psi(s_tr, s_te), float(np.mean(~np.isnan(s_tr))),\n                     float(np.mean(~np.isnan(s_te))), ks(np.r_[np.zeros(len(s_tr)), np.ones(len(s_te))],\n                                                          np.r_[np.nan_to_num(s_tr, nan=-1e18), np.nan_to_num(s_te, nan=-1e18)])))\n    res = pl.DataFrame(rows, schema=["feature", "ks_target", "psi_test", "cov_train", "cov_test", "ks_train_vs_test"],\n                       orient="row").sort("ks_target", descending=True)\n    with pl.Config(tbl_rows=200, fmt_str_lengths=40, float_precision=4):\n        print(f"\\n=== {tag}: univariate KS vs target, drift train->test")\n        print(res)\n    res.write_csv(OUT / f"{tag}_ks.csv")\n    top = res["feature"].head(top_plot).to_list()\n    n = len(top); ncol = 4; nrow = (n + ncol - 1) // ncol\n    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), squeeze=False)\n    for ax, c in zip(axes.ravel(), top):\n        s = tr.select(c, "target")\n        if s[c].dtype == pl.String:\n            g = s.group_by(c).agg(r=pl.col("target").mean(), n=pl.len()).sort(c)\n            ax.bar(g[c].cast(str), g["r"]); ax.tick_params(axis="x", rotation=45)\n        else:\n            s = s.with_columns(b=pl.when(pl.col(c).is_null() | pl.col(c).is_nan()).then(-1)\n                               .otherwise(pl.col(c).rank("ordinal") * 10 // (pl.col(c).is_not_nan().sum() + 1)))\n            g = s.group_by("b").agg(r=pl.col("target").mean(), n=pl.len()).sort("b")\n            ax.bar(g["b"].cast(str).str.replace("-1", "NA"), g["r"], color=["grey" if v == -1 else "C0" for v in g["b"]])\n        ax.axhline(y.mean(), color="r", ls="--", lw=1)\n        ax.set_title(f"{c}\\nKS={res.filter(pl.col(\'feature\') == c)[\'ks_target\'][0]:.3f}", fontsize=9)\n    for ax in axes.ravel()[n:]:\n        ax.axis("off")\n    fig.suptitle(f"{tag}: bad rate by decile (NA=grey, red=base rate)")\n    plt.tight_layout(); plt.savefig(OUT / f"{tag}_deciles.png", dpi=80); plt.close()\n    return res\n\n\n# ---- residual analysis against the v1 model (results/v1 OOF; no model is trained here) ----\ndef v1_oof():\n    """Rank-average of v1 GBDT OOF predictions, aligned to the train order of features()."""\n    import numpy as np\n    from scipy.stats import rankdata\n    s = np.mean([rankdata(np.load(ROOT / "results" / "v1" / f"oof_{n}.npy")) for n in ("lgb", "xgb", "cat")], 0) / 70000\n    # align by application_id: the row order of a rebuilt feature matrix is not guaranteed to match the v1 run\n    v1 = pl.read_parquet(PQ / "v1_train_order.parquet").with_columns(s=pl.Series(s))\n    cur = features().filter(pl.col("is_train") == 1).select("application_id")\n    return cur.join(v1, on="application_id", how="left", maintain_order="left")["s"].to_numpy()\n\n\ndef calibrate(y, s, bins=50):\n    """Score -> probability via bad rate in quantile bins of s (monotone enough for residual tests)."""\n    import numpy as np\n    b = np.minimum((np.argsort(np.argsort(s)) * bins) // len(s), bins - 1)\n    rate = np.bincount(b, weights=y, minlength=bins) / np.bincount(b, minlength=bins)\n    return rate[b]\n\n\ndef qbin(x, bins=10):\n    """Quantile bin id (0..bins-1); NaN -> bins (own bin). Ties stay together."""\n    import numpy as np\n    x = np.asarray(x, float)\n    out = np.full(len(x), bins)\n    m = ~np.isnan(x)\n    if m.sum() == 0:\n        return out\n    qs = np.unique(np.quantile(x[m], np.linspace(0, 1, bins + 1))[1:-1])\n    out[m] = np.searchsorted(qs, x[m], side="right")\n    return out\n\n\ndef resid_test(y, p, x, bins=10):\n    """Does x explain what p (calibrated v1 OOF) missed? Returns (chi2/df, max |O/E-1| over bins >=500 rows, cross-fit dKS).\n    chi2 uses binomial variance per bin; dKS = KS gain of logit(p)+log(O/E)_bin with bin effects fit on the other half."""\n    import numpy as np\n    b = qbin(x, bins)\n    o = np.bincount(b, weights=y, minlength=bins + 1)\n    e = np.bincount(b, weights=p, minlength=bins + 1)\n    v = np.bincount(b, weights=p * (1 - p), minlength=bins + 1)\n    n = np.bincount(b, minlength=bins + 1)\n    k = n > 0\n    chi = float((((o - e) ** 2)[k] / np.maximum(v[k], 1e-9)).sum() / max(k.sum() - 1, 1))\n    big = n >= 500\n    lift = float(np.abs(o[big] / np.maximum(e[big], 1e-9) - 1).max()) if big.any() else 0.0\n    rng = np.random.default_rng(0)\n    half = rng.random(len(y)) < 0.5\n    lp = np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))\n    adj = np.zeros(len(y))\n    for fit in (half, ~half):\n        of = np.bincount(b[fit], weights=y[fit], minlength=bins + 1) + 5\n        ef = np.bincount(b[fit], weights=p[fit], minlength=bins + 1) + 5\n        adj[~fit] = np.log(of / ef)[b[~fit]]\n    return chi, lift, ks(y, lp + adj) - ks(y, lp)\n\n\ndef screen(cands: pl.DataFrame, tag: str, show: int = 40):\n    """Screen candidate features (customer_id + columns) against v1: univariate KS, residual chi2/dKS vs v1 OOF,\n    PSI train->test, and max |Spearman| with any v1 feature (redundancy). Writes out/{tag}_screen.csv."""\n    import numpy as np\n    f = features()\n    tr = f.filter(pl.col("is_train") == 1)\n    te = f.filter(pl.col("is_train") == 0)\n    y = tr["target"].to_numpy()\n    p = calibrate(y, v1_oof())\n    ctr = tr.select("customer_id").join(cands, on="customer_id", how="left", maintain_order="left")\n    cte = te.select("customer_id").join(cands, on="customer_id", how="left", maintain_order="left")\n    ID = {"application_id", "customer_id", "decision_datetime", "target", "is_train"}\n    v1 = [c for c in f.columns if c not in ID and f[c].dtype != pl.String]\n    sub = np.random.default_rng(0).choice(len(y), 15000, replace=False)\n    V = tr[sub].select(v1).with_columns(pl.all().cast(pl.Float64).rank()).to_numpy()\n    rows = []\n    for c in cands.columns:\n        if c == "customer_id":\n            continue\n        x = ctr[c].cast(pl.Float64).fill_nan(None).to_numpy().astype(float)\n        xt = cte[c].cast(pl.Float64).fill_nan(None).to_numpy().astype(float)\n        chi, lift, dks = resid_test(y, p, x)\n        xs = pl.Series(x[sub]).rank().to_numpy()\n        m = ~np.isnan(xs)\n        red = 0.0\n        if m.sum() > 100:\n            Vm = V[m]; xm = xs[m]\n            ok = ~np.isnan(Vm)\n            # pairwise-complete spearman via ranks (approx: drop rows with NaN per column)\n            cors = []\n            for j in range(V.shape[1]):\n                k = ok[:, j]\n                if k.sum() > 100 and np.std(Vm[k, j]) > 0 and np.std(xm[k]) > 0:\n                    cors.append(abs(np.corrcoef(xm[k], Vm[k, j])[0, 1]))\n            red = max(cors) if cors else 0.0\n        rows.append((c, ks_nan(y, x), chi, dks, float(np.mean(~np.isnan(x))), psi(x, xt), red))\n    r = pl.DataFrame(rows, schema=["feature", "ks", "chi2_df", "dKS_cf", "cov", "psi", "max_rho_v1"], orient="row")\n    r = r.sort("chi2_df", descending=True)\n    with pl.Config(tbl_rows=show, float_precision=4, fmt_str_lengths=45):\n        print(f"\\n=== {tag}: candidates vs v1 residual (noise: chi2/df p95~1.2, dKS p95~+0.002)")\n        print(r.head(show))\n    r.write_csv(OUT / f"{tag}_screen.csv")\n    return r\n\n\ndef v2_oof():\n    """Equal rank-blend of the six v2 GBDT OOF predictions (results/v2). The v2 run used the same build_all code\n    as the local features(), and the row order was verified (lgb OOF KS 0.5118 matches the notebook)."""\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / "results" / "v2"\n    return np.mean([rankdata(np.load(R / f"oof_{n}.npy")) for n in ("lgb", "lgb_s", "xgb", "xgb_s", "cat", "cat_s")], 0) / 70000\n\n\ndef v5_oof():\n    """The submitted v5 blend (weights [4,0,0,1,9,3,0,13]/30 reconstructed from the run) as OOF ranks in features() order."""\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / "results" / "v5"\n    w = {"lgb": 4, "xgb_s": 1, "cat": 9, "cat_s": 3, "tabm": 13}\n    return sum(v * rankdata(np.load(R / f"oof_{n}.npy")) for n, v in w.items()) / 30 / 70000\n\n\ndef v6_oof():\n    """The submitted v6 blend (greedy weights [3,1,0,0,2,1,6,10,7]/30) as OOF ranks in features() order."""\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / "results" / "v6"\n    w = {"lgb": 3, "lgb_s": 1, "cat": 2, "cat_s": 1, "tabpfn": 6, "tabm": 10, "tabm_p": 7}\n    return sum(v * rankdata(np.load(R / f"oof_{n}.npy")) for n, v in w.items()) / 30 / 70000\n', 'scripts/run_dsc_s42.sh': '#!/usr/bin/env bash\n# DSC0152 v66 on valid2 folds: five folds, seed 42, inner stopping and refit.\nset -euo pipefail\n\nSCRIPT_DIR=$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)\nREPOSITORY=$(cd -- "$SCRIPT_DIR/.." && pwd)\nPYTHON="${LOGIKA_PYTHON:-$REPOSITORY/.venv/bin/python}"\nexport OMP_NUM_THREADS=4\n\nif [[ ! -x "$PYTHON" ]]; then\n    echo "Python environment missing: $PYTHON. See environments/README.md." >&2\n    exit 1\nfi\n\ncd "$REPOSITORY/scripts/dsc0152"\n\nrun_cpu() {\n    taskset -c "${LOGIKA_CPU_AFFINITY:-0-11}" nice -n 10 "$PYTHON" "$@"\n}\n\nif [[ ! -f agas/shared/cache/features_cache.pkl ]]; then\n    run_cpu -c \'\nimport os\nimport sys\nsys.path.insert(0, "agas/shared")\nimport paths\nimport features\npaths.CACHE.mkdir(parents=True, exist_ok=True)\nframe = features.build()\ntemporary = str(paths.FEATURES_CACHE) + ".tmp"\nframe.to_pickle(temporary)\nos.replace(temporary, paths.FEATURES_CACHE)\nprint("features", frame.shape, flush=True)\n\'\nfi\n\nif [[ ! -f agas/shared/cache/extra_cache.pkl ]]; then\n    run_cpu agas/shared/build_extra.py bureau,repayment,previous,account,ecommerce,interactions,gradenorm,rxtra,bxtra,axtra,ecommerce2,consistency,dataquality,stress\nfi\n\nrun_cpu agas/v66/training/train_submission_v66.py \\\n    --folds 5 --fold-seeds 42 --lgb-seeds 0,1,2,3,4 \\\n    --valid2-folds --inner-stop --n-jobs 4 --out-prefix v66s42\necho DSC_DONE\n', 'artifacts/optimization/continuation_25okt/recipe_merge.json': '{\n "name": "merge_R2DK",\n "fold_scheme": "StratifiedKFold(5, shuffle, 42) over id-sorted train (eda/valid2.outer)",\n "dsc_npz": "results/dsc0152/agas/v66/output/preds_v66s42.npz",\n "output_dir": "results/v71/merge_pipeline_R2DK",\n "families": {\n  "K64": {\n   "what": "Kumo strict (utama): cap300@28k + main_small, 2 seeds, pools fit on outer-train",\n   "steps": [\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 219_build_outer_pools.py 0",\n     "outputs": [\n      "results/v60/strict_tfm/pool_outer_f0/manifest.json"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 219_build_outer_pools.py 1",\n     "outputs": [\n      "results/v60/strict_tfm/pool_outer_f1/manifest.json"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 219_build_outer_pools.py 2",\n     "outputs": [\n      "results/v60/strict_tfm/pool_outer_f2/manifest.json"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 219_build_outer_pools.py 3",\n     "outputs": [\n      "results/v60/strict_tfm/pool_outer_f3/manifest.json"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 219_build_outer_pools.py 4",\n     "outputs": [\n      "results/v60/strict_tfm/pool_outer_f4/manifest.json"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 0 --member cap300_ctx28k_s2026",\n     "outputs": [\n      "results/v64/strict_variant/outer_f0/cap300_ctx28k_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 1 --member cap300_ctx28k_s2026",\n     "outputs": [\n      "results/v64/strict_variant/outer_f1/cap300_ctx28k_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 2 --member cap300_ctx28k_s2026",\n     "outputs": [\n      "results/v64/strict_variant/outer_f2/cap300_ctx28k_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 3 --member cap300_ctx28k_s2026",\n     "outputs": [\n      "results/v64/strict_variant/outer_f3/cap300_ctx28k_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 4 --member cap300_ctx28k_s2026",\n     "outputs": [\n      "results/v64/strict_variant/outer_f4/cap300_ctx28k_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 0 --member cap300_ctx28k_s2027",\n     "outputs": [\n      "results/v64/strict_variant/outer_f0/cap300_ctx28k_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 1 --member cap300_ctx28k_s2027",\n     "outputs": [\n      "results/v64/strict_variant/outer_f1/cap300_ctx28k_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 2 --member cap300_ctx28k_s2027",\n     "outputs": [\n      "results/v64/strict_variant/outer_f2/cap300_ctx28k_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 3 --member cap300_ctx28k_s2027",\n     "outputs": [\n      "results/v64/strict_variant/outer_f3/cap300_ctx28k_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v64_cap300_ctx28k.json --out ../results/v64/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 4 --member cap300_ctx28k_s2027",\n     "outputs": [\n      "results/v64/strict_variant/outer_f4/cap300_ctx28k_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 0 --member main_small_s2026",\n     "outputs": [\n      "results/v62/strict_variant/outer_f0/main_small_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 1 --member main_small_s2026",\n     "outputs": [\n      "results/v62/strict_variant/outer_f1/main_small_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 2 --member main_small_s2026",\n     "outputs": [\n      "results/v62/strict_variant/outer_f2/main_small_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 3 --member main_small_s2026",\n     "outputs": [\n      "results/v62/strict_variant/outer_f3/main_small_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 4 --member main_small_s2026",\n     "outputs": [\n      "results/v62/strict_variant/outer_f4/main_small_s2026.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 0 --member main_small_s2027",\n     "outputs": [\n      "results/v62/strict_variant/outer_f0/main_small_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 1 --member main_small_s2027",\n     "outputs": [\n      "results/v62/strict_variant/outer_f1/main_small_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 2 --member main_small_s2027",\n     "outputs": [\n      "results/v62/strict_variant/outer_f2/main_small_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 3 --member main_small_s2027",\n     "outputs": [\n      "results/v62/strict_variant/outer_f3/main_small_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 224_strict_tfm_variant.py --protocol ../artifacts/optimization/continuation_24okt/protocol_v62_ctx16k.json --out ../results/v62/strict_variant --pool-root ../results/v60/strict_tfm --stage outer --fold 4 --member main_small_s2027",\n     "outputs": [\n      "results/v62/strict_variant/outer_f4/main_small_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 221_evaluate_recipe_candidates.py results/v64/strict_variant::cap300_ctx28k_s2026 results/v64/strict_variant::cap300_ctx28k_s2027 results/v62/strict_variant::main_small_s2026 results/v62/strict_variant::main_small_s2027 --tag K64_2seed",\n     "outputs": [\n      "results/v60/candidates/K__K64_2seed/oof.parquet",\n      "results/v60/candidates/K__K64_2seed/submission.csv"\n     ]\n    }\n   ]\n  },\n  "D": {\n   "what": "DSC0152 v66 ext0.5|tm30|lgb_lean, 5-fold seed 42, inner early stop + refit",\n   "steps": [\n    {\n     "cmd": "bash ../scripts/run_dsc_s42.sh",\n     "outputs": [\n      "results/dsc0152/agas/v66/output/preds_v66s42.npz"\n     ]\n    }\n   ]\n  },\n  "KF": {\n   "what": "Kumo V5 + DSC0152 factor features (253)",\n   "steps": [\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage factor",\n     "outputs": [\n      "results/v71/teammate_s42/f0/factor.parquet",\n      "results/v71/teammate_s42/f1/factor.parquet",\n      "results/v71/teammate_s42/f2/factor.parquet",\n      "results/v71/teammate_s42/f3/factor.parquet",\n      "results/v71/teammate_s42/f4/factor.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage kumo --fold 0",\n     "outputs": [\n      "results/v71/teammate_s42/f0/kumo_fac.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage kumo --fold 1",\n     "outputs": [\n      "results/v71/teammate_s42/f1/kumo_fac.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage kumo --fold 2",\n     "outputs": [\n      "results/v71/teammate_s42/f2/kumo_fac.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage kumo --fold 3",\n     "outputs": [\n      "results/v71/teammate_s42/f3/kumo_fac.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage kumo --fold 4",\n     "outputs": [\n      "results/v71/teammate_s42/f4/kumo_fac.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 253_kumo_v5_factor.py --stage eval",\n     "outputs": [\n      "results/v71/teammate_s42/oof_kf.parquet",\n      "results/v71/teammate_s42/test_kf.parquet"\n     ]\n    }\n   ]\n  },\n  "KD": {\n   "what": "Kumo on DSC0152 features cap300@28k (254); needs D cache + KF factor",\n   "steps": [\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 0",\n     "outputs": [\n      "results/v71/teammate_s42/f0/kumo_dsc.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 1",\n     "outputs": [\n      "results/v71/teammate_s42/f1/kumo_dsc.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 2",\n     "outputs": [\n      "results/v71/teammate_s42/f2/kumo_dsc.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 3",\n     "outputs": [\n      "results/v71/teammate_s42/f3/kumo_dsc.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 4",\n     "outputs": [\n      "results/v71/teammate_s42/f4/kumo_dsc.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage eval",\n     "outputs": [\n      "results/v71/teammate_s42/oof_kd.parquet",\n      "results/v71/teammate_s42/test_kd.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 0 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f0/kumo_dsc_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 1 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f1/kumo_dsc_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 2 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f2/kumo_dsc_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 3 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f3/kumo_dsc_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage kumo --fold 4 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f4/kumo_dsc_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 254_kumo_dsc_features.py --stage eval --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/oof_kd_s2027.parquet",\n      "results/v71/teammate_s42/test_kd_s2027.parquet"\n     ]\n    }\n   ]\n  },\n  "R2": {\n   "what": "teman goal054: Kumo V5 ctx50k e8 + offset re_* + spline 25%, model seeds 2026 and 2027 averaged (R2)",\n   "steps": [\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 166_teammate_nested_features.py --out ../results/v56/teammate_features/outer_s42_f0 --partition outer --seed 42 --fold 0",\n     "outputs": [\n      "results/v56/teammate_features/outer_s42_f0/data/processed/application_features_v5.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 166_teammate_nested_features.py --out ../results/v56/teammate_features/outer_s42_f1 --partition outer --seed 42 --fold 1",\n     "outputs": [\n      "results/v56/teammate_features/outer_s42_f1/data/processed/application_features_v5.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 166_teammate_nested_features.py --out ../results/v56/teammate_features/outer_s42_f2 --partition outer --seed 42 --fold 2",\n     "outputs": [\n      "results/v56/teammate_features/outer_s42_f2/data/processed/application_features_v5.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 166_teammate_nested_features.py --out ../results/v56/teammate_features/outer_s42_f3 --partition outer --seed 42 --fold 3",\n     "outputs": [\n      "results/v56/teammate_features/outer_s42_f3/data/processed/application_features_v5.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 166_teammate_nested_features.py --out ../results/v56/teammate_features/outer_s42_f4 --partition outer --seed 42 --fold 4",\n     "outputs": [\n      "results/v56/teammate_features/outer_s42_f4/data/processed/application_features_v5.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage reverse --fold 0",\n     "outputs": [\n      "results/v71/teammate_s42/f0/reverse.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage reverse --fold 1",\n     "outputs": [\n      "results/v71/teammate_s42/f1/reverse.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage reverse --fold 2",\n     "outputs": [\n      "results/v71/teammate_s42/f2/reverse.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage reverse --fold 3",\n     "outputs": [\n      "results/v71/teammate_s42/f3/reverse.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage reverse --fold 4",\n     "outputs": [\n      "results/v71/teammate_s42/f4/reverse.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 0",\n     "outputs": [\n      "results/v71/teammate_s42/f0/kumo.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 0",\n     "outputs": [\n      "results/v71/teammate_s42/f0/post.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 1",\n     "outputs": [\n      "results/v71/teammate_s42/f1/kumo.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 1",\n     "outputs": [\n      "results/v71/teammate_s42/f1/post.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 2",\n     "outputs": [\n      "results/v71/teammate_s42/f2/kumo.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 2",\n     "outputs": [\n      "results/v71/teammate_s42/f2/post.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 3",\n     "outputs": [\n      "results/v71/teammate_s42/f3/kumo.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 3",\n     "outputs": [\n      "results/v71/teammate_s42/f3/post.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 4",\n     "outputs": [\n      "results/v71/teammate_s42/f4/kumo.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 4",\n     "outputs": [\n      "results/v71/teammate_s42/f4/post.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage eval",\n     "outputs": [\n      "results/v71/teammate_s42/oof.parquet",\n      "results/v71/teammate_s42/test.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 0 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f0/kumo_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 0 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f0/post_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 1 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f1/kumo_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 1 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f1/post_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 2 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f2/kumo_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 2 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f2/post_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 3 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f3/kumo_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 3 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f3/post_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage kumo --fold 4 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f4/kumo_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage post --fold 4 --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/f4/post_s2027.parquet"\n     ]\n    },\n    {\n     "cmd": "OMP_NUM_THREADS=4 taskset -c 0-11 nice -n 10 ../.venv/bin/python 251_teammate_recipe_s42.py --stage eval --seed 2027",\n     "outputs": [\n      "results/v71/teammate_s42/oof_s2027.parquet",\n      "results/v71/teammate_s42/test_s2027.parquet"\n     ]\n    }\n   ]\n  }\n },\n "blend": {\n  "members": [\n   "R2",\n   "D",\n   "K64"\n  ],\n  "weights": "equal, rank per outer cohort (OOF) / rank over test (submission); structure locked in protocol_252_merge.json (RDK primary), second Kumo seed in protocol_256_r_seed2.json (R2DK)",\n  "input_prefixes": [\n   "results/v71/teammate_s42/oof",\n   "results/v71/teammate_s42/test",\n   "results/dsc0152/agas/v66/output/preds_v66s42",\n   "results/v60/candidates/K__K64_2seed/"\n  ]\n }\n}\n'}
FILE_SHA256={'eda/260_merge_pipeline.py': '4f7cde06d9cf82439e473cbefe93ddd49d16578c5b2b09afec9cfcab4cbacf30', 'eda/250_merge_blend_existing.py': 'b53bb603c3d1bb8ac45a3b38ca25c4e298316ce5ed343a8f5f2bef7cd89518dd', 'eda/221_evaluate_recipe_candidates.py': '433489c2ff971f8becaf22e2cd7098d9a6a03cc12b9c7eb49a773bb827750af5', 'eda/251_teammate_recipe_s42.py': '8a63d3963f5dc7e0fda827926720cffb221d45d39d556a3cc7ca7a0602776155', 'eda/254_kumo_dsc_features.py': '4e44f9adc800e9601485aa4b47e265e2475a13143ea10f1ca3e28d74264563ff', 'eda/valid2.py': '3f1bfe40ae9e32d70577f537e6b7a29e2a103d7a423eb03026e62ab37d01f8fc', 'eda/common.py': '679d20a7beb3d5e3946232d1dcea66f21b871ab1a41c8879e15d3d5a615f92e7', 'scripts/run_dsc_s42.sh': 'ad1ef0a8d2e2c06eaef685d60f3d8f8ff55121a88b9ff409114989b462569e67', 'artifacts/optimization/continuation_25okt/recipe_merge.json': 'fc2e808150a9deb1c5d4201859627009e6a26b21440d8e37dd4861a2c15a7717'}
for name,source in FILES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==FILE_SHA256[name]
    assert (REPOSITORY/name).read_text()==source,name
print('Embedded files match repository',len(FILES))

## 4. Families and blend

Each family is rank-transformed inside each outer cohort (OOF) or over the 30,000 test rows (submission) and averaged with equal weight.

In [ ]:
eda=REPOSITORY/'eda'
if MODE=='fresh': subprocess.run([sys.executable,str(eda/'260_merge_pipeline.py'),'--stage','build'],cwd=eda,check=True,env=os.environ.copy())
subprocess.run([sys.executable,str(eda/'260_merge_pipeline.py'),'--stage','blend','--out',str(OUTPUT_DIR)],cwd=eda,check=True,env=os.environ.copy())

## 5. CSV validation

In [ ]:
submission=pl.read_csv(OUTPUT_DIR/'submission.csv')
sample=pl.read_csv(REPOSITORY/'data/sample_submission.csv')
assert submission.columns==['application_id','default_probability']
assert submission.height==sample.height==30000 and submission['application_id'].n_unique()==30000
assert submission['application_id'].to_list()==sample['application_id'].to_list()
p=submission['default_probability'].to_numpy()
assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
ref=json.loads((REPOSITORY/'results/v71/merge_pipeline_R2DK/manifest.json').read_text())
digest=hashlib.sha256((OUTPUT_DIR/'submission.csv').read_bytes()).hexdigest()
print('submission',digest,'reference',ref['outputs']['submission.csv'],'identical',digest==ref['outputs']['submission.csv'])
if MODE=='replay': assert digest==ref['outputs']['submission.csv']